1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working')
COMFY_DIR = ROOT / 'ComfyUI'

'''
Select the model and workflow storage locations.
設定 LTX 模型下載、Workflow 註冊表與引擎檔案的路徑。
'''
SCRATCH_DIR = Path('/tmp/ltx25')
MODEL_STORE = SCRATCH_DIR / 'ltx25_models'
FLOW_REGISTRY_PATH = ROOT / 'ltx25_local_flows.json'
COMFY_LOG = ROOT / 'comfyui_ltx25.log'
COMFY_PORT = 8188  # 8188、8189 等未使用的 Port
COMFY_REF = 'v0.45.0'

'''
Choose the model cache folder and apply it before downloads.
設定模型下載快取位置，並在下載前套用。
'''
HF_CACHE_DIR = SCRATCH_DIR / 'huggingface'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_XET_CACHE'] = str(HF_CACHE_DIR / 'xet')
NODE_CACHE_MODE = 'none'  # 'none', 'classic'
GPU_VRAM_RESERVE_GIB = 1  # 1、2，單位 GiB

'''
Require enough available host RAM before loading a large video workflow.
設定生成前主機／容器 RAM 的最低安全門檻。
'''
LTX_MIN_AVAILABLE_RAM_GIB = 6.0  # 例如 6.0、8.0 GiB
MODEL_TIER = 'q4km'  # 此版固定；切換需另備相容權重與 Workflow
FPS = 24
VAE_TILE_SIZE, VAE_TILE_OVERLAP = 256, 32
VAE_TEMPORAL_SIZE, VAE_TEMPORAL_OVERLAP = 16, 4

'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_LTX-2.5_影片生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = 2
LTX_OUTPUT_DIR = ROOT / 'ltx25_api_outputs'
os.environ['COMFY_BASE'] = f'http://127.0.0.1:{COMFY_PORT}'
os.environ['COMFY_DIR'] = str(COMFY_DIR)
os.environ['FLOW_REGISTRY_PATH'] = str(FLOW_REGISTRY_PATH)
os.environ['LTX25_COMFY_LOG'] = str(COMFY_LOG)
os.environ['LTX25_COMFY_PID'] = str(ROOT / 'comfyui_ltx25.pid')
os.environ['LTX25_OUTPUT_DIR'] = str(LTX_OUTPUT_DIR)

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= COMFY_PORT <= 65535 and COMFY_PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝套件


In [ ]:
'''
Install ComfyUI and the custom nodes used by the LTX-2.5 workflow.
安裝 LTX-2.5 Workflow 會用到的 ComfyUI 與自訂節點。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

repos = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY_DIR, COMFY_REF),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY_DIR / 'custom_nodes/ComfyUI-GGUF', None),
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git', COMFY_DIR / 'custom_nodes/ComfyUI-LTXVideo', None),
    ('https://github.com/kijai/ComfyUI-KJNodes.git', COMFY_DIR / 'custom_nodes/ComfyUI-KJNodes', None),
    ('https://github.com/yolain/ComfyUI-Easy-Use.git', COMFY_DIR / 'custom_nodes/ComfyUI-Easy-Use', None),
    ('https://github.com/SethRobinson/comfyui-workflow-to-api-converter-endpoint.git', COMFY_DIR / 'custom_nodes/workflow-to-api-converter', None),
]
UPDATE_CUSTOM_NODES = True
original_revisions = {}

'''
Install missing repositories and safely update existing ones.
安裝缺少的倉庫，並安全更新已存在的倉庫。
'''
for url, directory, ref in repos:
    if directory.exists() and not (directory / '.git').exists():
        raise RuntimeError(f'已有非 Git 資料夾：{directory}')
    if (directory / '.git').exists():
        original_revisions[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        command = ['git', '-c', 'advice.detachedHead=false', 'clone', '--depth', '1', '-q']

        '''
        Select the stable release for ComfyUI itself.
        ComfyUI 核心固定於穩定發行版。
        '''
        if ref:
            command += ['--branch', ref]
        subprocess.run(command + [url, str(directory)], check=True)
    if directory != COMFY_DIR and UPDATE_CUSTOM_NODES:
        if not subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip():
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:
            print(f'略過已有本機修改的外掛：{directory.name}')

subprocess.run(['git', '-C', str(COMFY_DIR), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY_DIR), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)

'''
Indicate whether an existing model server must be restarted.
標記已啟動的模型服務是否需要重新啟動。
'''
LTX_CODE_UPDATED = any(
    directory in original_revisions and original_revisions[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for _, directory, _ in repos
)
for _, directory, _ in repos:
    print(directory.name, subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip())

constraint = ROOT / 'ltx25_torch_constraint.txt'

'''
Save the interpreter-specific compatibility limits.
儲存目前 Python 環境適用的相容性限制。
'''
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

for _, directory, _ in repos:
    requirement = directory / 'requirements.txt'
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'Pillow', 'ipywidgets'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.8.2'], check=True)

legacy = COMFY_DIR / 'custom_nodes/ComfyUI-MultiGPU'
if legacy.is_dir():
    legacy.rename(ROOT / 'ComfyUI-MultiGPU.disabled')

print('ComfyUI installed:', COMFY_DIR)
print('Torch:', torch.__version__)
print('✓ 第 2 節完成：LTX-2.5 環境與套件已安裝。')


3. 設定環境


In [ ]:
'''
Import the packages required by the notebook.
匯入 Notebook 所需套件。
'''
import importlib.util
import os
import shutil
import subprocess
import time
from pathlib import Path
import ipywidgets as widgets
import requests
import torch
from IPython.display import FileLink, Video, display
from PIL import Image
MODEL_STORE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_XET_CACHE'] = str(HF_CACHE_DIR / 'xet')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

'''
Require Kaggle's two-GPU runtime before loading the workflow.
載入 Workflow 前，確認 Kaggle 已啟用雙 GPU。
'''
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'
for index in range(2):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

print('✓ 第 3 節完成：雙 GPU 裝置與環境設定已檢查。')


4. 模型與 Workflow


In [ ]:
'''
Fetch only the model files used by this Q4_K_M workflow.
只下載 Q4_K_M 雙階段 Workflow 真正需要的模型。
'''
import json
import shutil
import wave
from huggingface_hub import hf_hub_download
from PIL import Image
FLOW_REGISTRY = {
    "schema_version": 25,
    "models": {
        "transformers": {"q4km": {"filename": "LTX-2.5-Distilled-Q4_K_M.gguf"}},
        "text_encoder": {"basename": "gemma4-12b-ltx25-w4a8-v2.safetensors"},
        "video_vae": {"filename": "vae/ltx-2.5-video-vae-conv-bf16.safetensors"},
        "audio_vae": {"filename": "vae/ltx-2.5-audio-vae-bf16.safetensors"},
        "spatial_upscaler": {"filename": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"},
    },
    "workflows": {"t2v_i2v": {"local_name": "LTX25_T2V_I2V.json"}},
}
model_files = [
    ("realrebelai/LTX-2.5_GGUFs", "LTX-2.5-Distilled-Q4_K_M.gguf", "diffusion_models"),
    ("realrebelai/Rebels_w4a8s", "LTX/ENCODERS/gemma4-12b-ltx25-w4a8-v2.safetensors", "text_encoders"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-video-vae-conv-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", "latent_upscale_models"),
]

'''
Link scratch weights into the model folders expected by ComfyUI.
透過 symlink 將暫存權重連到 ComfyUI 預期的模型資料夾。
'''
for repo, filename, folder in model_files:
    target = COMFY_DIR / "models" / folder / Path(filename).name
    stored = MODEL_STORE / folder / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    stored.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and not target.is_symlink() and target.stat().st_size > 1_000_000:
        if not stored.exists():
            shutil.move(str(target), str(stored))
        elif target.stat().st_size == stored.stat().st_size:
            target.unlink()
        else:
            print("Existing model retained:", target.name)
            continue
    if not stored.is_file() or stored.stat().st_size <= 1_000_000:
        stored = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(MODEL_STORE / folder)))
    if target.is_symlink() or target.is_file():
        target.unlink()
    target.symlink_to(stored)
    print("Model:", target.name, "->", stored)

gguf_path = COMFY_DIR / "models/diffusion_models/LTX-2.5-Distilled-Q4_K_M.gguf"
multigpu_dir = COMFY_DIR / "models/unet_gguf"
multigpu_dir.mkdir(parents=True, exist_ok=True)
multigpu_link = multigpu_dir / gguf_path.name
if multigpu_link.is_symlink() and not multigpu_link.exists():
    multigpu_link.unlink()

if not multigpu_link.exists():
    multigpu_link.symlink_to(gguf_path)

workflow_file = hf_hub_download(
    repo_id="realrebelai/LTX-2.5_GGUFs",
    filename="LTX 2.5 GGUF (WORKFLOW).json",
    revision="112436f97aaf99ce13ecb7b7eca7e2f6c128d3ec",
    local_dir=str(SCRATCH_DIR / ".hf_workflow_stage"),
)
ui = json.loads(Path(workflow_file).read_text(encoding="utf-8"))

'''
Substitute model filenames throughout the UI workflow, including subgraphs.
在完整 UI Workflow（含子圖）中替換模型檔名。
'''
replacements = {
    "gguf": gguf_path.name,
    "text_encoder": FLOW_REGISTRY["models"]["text_encoder"]["basename"],
    "video_vae": Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name,
    "audio_vae": Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name,
    "upscaler": Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name,
}

def replace_names(value):
    '''
    Keep this transformation recursive because the workflow contains subgraphs.
    Workflow 包含巢狀子圖，因此需遞迴處理名稱。
    '''
    if isinstance(value, dict):
        return {key: replace_names(item) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_names(item) for item in value]
    if not isinstance(value, str):
        return value
    lower = value.lower()
    if lower.endswith(".gguf") and "ltx" in lower:
        return replacements["gguf"]
    if "gemma4" in lower and lower.endswith(".safetensors") and "enhancer" not in lower:
        return replacements["text_encoder"]
    if "video-vae" in lower and lower.endswith(".safetensors"):
        return replacements["video_vae"]
    if "audio-vae" in lower and lower.endswith(".safetensors"):
        return replacements["audio_vae"]
    if "upscaler" in lower and lower.endswith(".safetensors"):
        return replacements["upscaler"]
    return value

INPUT_DIR = COMFY_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
Image.new("RGB", (64, 64)).save(INPUT_DIR / "__ltx25_placeholder__.png")
with wave.open(str(INPUT_DIR / "__ltx25_placeholder__.wav"), "wb") as output_audio:
    output_audio.setnchannels(1)
    output_audio.setsampwidth(2)
    output_audio.setframerate(16000)
    output_audio.writeframes(b"\x00\x00" * 16000)

'''
Replace the sample filenames in nested LoadImage and LoadAudio nodes.
將子圖內的範例圖片與音訊名稱改成實際存在的預設檔。
'''
def patch_placeholders(graph):
    for node in graph.get("nodes", []):
        values = node.get("widgets_values")
        if node.get("type") == "LoadImage" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.png"
        if node.get("type") == "LoadAudio" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.wav"
    for child in graph.get("definitions", {}).get("subgraphs", []):
        patch_placeholders(child)

ui = replace_names(ui)
patch_placeholders(ui)
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows/LTX25-Kaggle"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
(WORKFLOW_DIR / "LTX25_T2V_I2V.json").write_text(json.dumps(ui, ensure_ascii=False), encoding="utf-8")
FLOW_REGISTRY_PATH.write_text(json.dumps(FLOW_REGISTRY, ensure_ascii=False), encoding="utf-8")
print("Models and workflow ready.")
FLF_WORKFLOW_PATH = WORKFLOW_DIR / 'LTX25_FLF2V.json'
if not FLF_WORKFLOW_PATH.is_file():
    flf_url = 'https://raw.githubusercontent.com/UCHIJ/LTX2.5-first_to_last_frame/main/LTX%202.5-first_frame_to_last_frame.json'
    flf_response = requests.get(flf_url, timeout=90)
    flf_response.raise_for_status()
    flf_ui = flf_response.json()
    flf_nodes = list(flf_ui.get('nodes', []))
    for subgraph in flf_ui.get('definitions', {}).get('subgraphs', []):
        flf_nodes.extend(subgraph.get('nodes', []))
    flf_images = [node for node in flf_nodes if node.get('type') == 'LoadImage']
    flf_guides = [node for node in flf_nodes if node.get('type') == 'LTXVAddGuide']
    if len(flf_images) < 2 or len(flf_guides) < 4:
        raise RuntimeError('FLF2V Workflow does not contain two images and two-stage guides.')
    patch_placeholders(flf_ui)
    FLF_WORKFLOW_PATH.write_text(json.dumps(flf_ui, ensure_ascii=False), encoding='utf-8')

print('FLF2V workflow ready:', FLF_WORKFLOW_PATH.name)

'''
Install the original workflow conversion and output-handling engine locally.
在 Kaggle 本機建立原版 Workflow 轉換與輸出引擎。
'''
ENGINE_CODE = '\nimport asyncio\nimport io\nimport math\nimport re\nimport wave\nimport json\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport requests\nfrom PIL import Image\n\nCOMFY_BASE = os.environ.get("COMFY_BASE", "http://127.0.0.1:8188").rstrip("/")\nCOMFY_DIR = Path(os.environ.get("COMFY_DIR", "/kaggle/working/ComfyUI"))\nFLOW_REGISTRY_PATH = Path(os.environ.get("FLOW_REGISTRY_PATH", "/kaggle/working/ltx25_local_flows.json"))\nWORKFLOW_DIR = Path(os.environ.get(\n    "WORKFLOW_DIR",\n    str(COMFY_DIR / "user/default/workflows/LTX25-Kaggle"),\n))\n\nif not FLOW_REGISTRY_PATH.is_file():\n    raise RuntimeError(f"Flow registry missing: {FLOW_REGISTRY_PATH}")\n\nFLOW_REGISTRY = json.loads(FLOW_REGISTRY_PATH.read_text(encoding="utf-8"))\n\nSTACK_SCHEMA_VERSION = int(FLOW_REGISTRY.get("schema_version", 0))\n\nMODEL_FILES = {\n    tier: item["filename"]\n    for tier, item in FLOW_REGISTRY["models"]["transformers"].items()\n}\nPRIMARY_MODEL_TIER = os.environ.get("LTX25_PRIMARY_MODEL_TIER", "q4km").lower()\nFALLBACK_MODEL_TIERS = [\n    x.strip().lower()\n    for x in os.environ.get("LTX25_FALLBACK_MODEL_TIERS", "").split(",")\n    if x.strip()\n]\n\nVAE_TILE_SIZE = int(os.environ.get("LTX25_VAE_TILE_SIZE", "512"))\nVAE_TILE_OVERLAP = int(os.environ.get("LTX25_VAE_TILE_OVERLAP", "64"))\nVAE_TEMPORAL_SIZE = int(os.environ.get("LTX25_VAE_TEMPORAL_SIZE", "32"))\nVAE_TEMPORAL_OVERLAP = int(os.environ.get("LTX25_VAE_TEMPORAL_OVERLAP", "8"))\n\nCOMFY_LOG_PATH = Path(os.environ.get("LTX25_COMFY_LOG", "/kaggle/working/comfyui_ltx25.log"))\nCOMFY_PID_PATH = Path(os.environ.get("LTX25_COMFY_PID", "/kaggle/working/comfyui_ltx25.pid"))\nTEXT_ENCODER = Path(FLOW_REGISTRY["models"]["text_encoder"]["basename"]).name\nVIDEO_VAE = Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name\nAUDIO_VAE = Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name\nUPSCALER = Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name\n\nT2V_I2V_WORKFLOW = WORKFLOW_DIR / FLOW_REGISTRY["workflows"]["t2v_i2v"]["local_name"]\nFLF2V_WORKFLOW = WORKFLOW_DIR / "LTX25_FLF2V.json"\n\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}\nAUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".ogg"}\nMAX_IMAGE_BYTES = 15 * 1024 * 1024\nMAX_AUDIO_BYTES = 32 * 1024 * 1024\n\nJOBS = {}\nJOBS_LOCK = threading.Lock()\nEXECUTOR = ThreadPoolExecutor(max_workers=1)\n\n\n\ndef job_update(job_key, **fields):\n    # job_key is the dictionary key; fields may safely contain "job_id".\n    with JOBS_LOCK:\n        JOBS.setdefault(job_key, {}).update(fields)\n\n\ndef _comfy_http_error(method, path, response):\n    try:\n        body = response.json()\n        detail = json.dumps(body, ensure_ascii=False, indent=2)\n    except Exception:\n        detail = response.text[:12000]\n    raise RuntimeError(\n        f"ComfyUI {method} {path} failed: HTTP {response.status_code}\\n{detail}"\n    )\n\n\ndef comfy_get(path, **kwargs):\n    response = requests.get(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("GET", path, response)\n    return response\n\n\ndef comfy_post(path, **kwargs):\n    response = requests.post(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("POST", path, response)\n    return response\n\n\ndef get_gguf_loader_choices():\n    # Read only the GPU-selectable GGUF loader exposed by this notebook.\n    # 只檢查本 Notebook 註冊的雙 GPU GGUF Loader。\n    name = \'LTX25GGUFLoaderGPU\'\n    try:\n        spec = comfy_get(f\'/object_info/{name}\', timeout=30).json()[name]\n        return True, spec[\'input\'][\'required\'][\'unet_name\'][0], spec\n    except Exception as exc:\n        return False, [], {\'error\': str(exc)}\n\n\nKEEP_MODELS_RESIDENT = False\n\n\ndef free_comfy():\n    try:\n        comfy_post(\n            "/free",\n            json={"unload_models": True, "free_memory": True},\n            timeout=60,\n        )\n    except Exception:\n        pass\n\n\ndef wait_comfy_ready(timeout=120):\n    deadline = time.time() + timeout\n    last = None\n    while time.time() < deadline:\n        try:\n            response = comfy_get("/system_stats", timeout=3)\n            if response.ok:\n                return True\n        except Exception as exc:\n            last = exc\n        time.sleep(1)\n    raise RuntimeError(f"ComfyUI is not ready: {last}")\n\n\ndef validate_dimensions(width, height):\n    width = int(width)\n    height = int(height)\n    if width < 256 or height < 256:\n        raise ValueError("width and height must both be >= 256")\n    if width % 32 != 0 or height % 32 != 0:\n        raise ValueError("width and height must both be divisible by 32")\n    return width, height\n\n\ndef save_upload_bytes(data, suffix, prefix):\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = f"{prefix}_{uuid.uuid4().hex}{suffix.lower()}"\n    path = input_dir / name\n    path.write_bytes(data)\n    return name, path\n\n\ndef ensure_placeholder_image():\n    """\n    ComfyUI validates LoadImage nodes before execution even when the workflow\n    later disables image conditioning. Some downloaded UI workflows contain\n    stale example filenames such as Test__00139_.png.\n\n    Keep one valid tiny placeholder in ComfyUI/input and patch unused LoadImage\n    nodes to it. The workflow\'s use-image switch still controls conditioning.\n    """\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n\n    name = "__ltx25_placeholder__.png"\n    path = input_dir / name\n\n    if not path.is_file():\n        Image.new("RGB", (64, 64), (0, 0, 0)).save(path, format="PNG")\n\n    return name\n\n\ndef ensure_placeholder_audio():\n    """Create a valid one-second silent WAV for unused LoadAudio validation."""\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = "__ltx25_placeholder__.wav"\n    path = input_dir / name\n    if not path.is_file():\n        with wave.open(str(path), "wb") as wf:\n            wf.setnchannels(1)\n            wf.setsampwidth(2)\n            wf.setframerate(16000)\n            wf.writeframes(b"\\x00\\x00" * 16000)\n    return name\n\n\ndef validate_image_bytes(data, suffix):\n    if not data:\n        raise ValueError("Image is empty.")\n    if len(data) > MAX_IMAGE_BYTES:\n        raise ValueError(f"Image exceeds {MAX_IMAGE_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in IMAGE_EXTENSIONS:\n        raise ValueError(f"Unsupported image extension: {suffix}")\n    try:\n        with Image.open(io.BytesIO(data)) as image:\n            image.verify()\n    except Exception as exc:\n        raise ValueError("Invalid/corrupted image.") from exc\n\n\ndef ffprobe_audio_bytes(data, suffix):\n    if not data:\n        raise ValueError("Audio is empty.")\n    if len(data) > MAX_AUDIO_BYTES:\n        raise ValueError(f"Audio exceeds {MAX_AUDIO_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in AUDIO_EXTENSIONS:\n        raise ValueError(f"Unsupported audio extension: {suffix}")\n\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n        tmp.write(data)\n        path = tmp.name\n\n    try:\n        result = subprocess.run(\n            [\n                "ffprobe", "-v", "error",\n                "-select_streams", "a:0",\n                "-show_entries", "stream=codec_name,profile:format=duration",\n                "-of", "json",\n                path,\n            ],\n            capture_output=True,\n            text=True,\n            timeout=30,\n        )\n        if result.returncode != 0:\n            raise ValueError("ffprobe could not decode the audio.")\n        payload = json.loads(result.stdout or "{}")\n        streams = payload.get("streams") or []\n        if not streams:\n            raise ValueError("No audio stream found.")\n        duration = float((payload.get("format") or {}).get("duration") or 0)\n        if duration <= 0:\n            raise ValueError("Could not determine audio duration.")\n        return {\n            "codec": streams[0].get("codec_name"),\n            "profile": streams[0].get("profile"),\n            "duration": duration,\n        }\n    finally:\n        try:\n            os.remove(path)\n        except OSError:\n            pass\n\n\ndef _norm_label(value):\n    text = str(value or "").strip().lower().replace("_", " ")\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\nALIASES = {\n    "prompt": {"prompt", "positive prompt", "prompt positive", "text prompt", "prompt text", "main prompt", "positive text"},\n    "width": {"width", "video width", "output width"},\n    "height": {"height", "video height", "output height"},\n    "seconds": {"duration", "duration in seconds", "seconds", "video duration"},\n    "fps": {"fps", "frame rate", "frames per second"},\n    "seed": {"seed", "noise seed"},\n    "image_strength": {"img strength", "image strength"},\n    "video_vae": {"video vae"},\n    "audio_vae": {"audio vae"},\n    "text_encoder": {"text encoder", "encoder name", "clip name"},\n    "upscaler": {"upscale model", "upscaler", "spatial upscaler"},\n    "text_to_video": {"text to video", "t2v"},\n    "use_image": {"use image input", "use image", "image input"},\n    "use_audio": {"use custom audio", "custom audio", "use audio", "audio input"},\n    "prompt_enhance": {"enhance positive prompt", "prompt enhance", "enhance prompt"},\n    "tile_size": {"tile size"},\n    "tile_overlap": {"tile overlap"},\n    "temporal_size": {"temporal size"},\n    "temporal_overlap": {"temporal overlap"},\n    "model": {"ltx model", "diffusion model", "unet", "unet name", "model checkpoint"},\n}\nALIASES = {k: {_norm_label(x) for x in v} for k, v in ALIASES.items()}\n\n\ndef _matches_field_label(field, label):\n    label = _norm_label(label)\n    if label in ALIASES[field]:\n        return True\n    if field == "prompt" and "prompt" in label:\n        blocked = ("negative", "enhanc", "system", "instruction", "translated", "output")\n        return not any(word in label for word in blocked)\n    return False\n\n\ndef _iter_all_nodes(ui):\n    if not isinstance(ui, dict):\n        return\n    for node in ui.get("nodes", []):\n        if isinstance(node, dict):\n            yield ui, node\n    definitions = ui.get("definitions")\n    subgraphs = definitions.get("subgraphs", []) if isinstance(definitions, dict) else []\n    for subgraph in subgraphs:\n        if isinstance(subgraph, dict):\n            yield from _iter_all_nodes(subgraph)\n\n\ndef _remove_ui_link(owner, link_id):\n    links = owner.get("links")\n    if not isinstance(links, list) or link_id is None:\n        return\n    kept = []\n    for link in links:\n        if isinstance(link, list) and link and link[0] == link_id:\n            continue\n        if isinstance(link, dict) and link.get("id") == link_id:\n            continue\n        kept.append(link)\n    owner["links"] = kept\n\n\ndef _widget_inputs(node):\n    return [x for x in (node.get("inputs") or []) if isinstance(x, dict) and "widget" in x]\n\n\ndef _widget_index(node, input_name):\n    for i, inp in enumerate(_widget_inputs(node)):\n        if inp.get("name") == input_name:\n            return i\n    return None\n\n\ndef _coerce_widget_value(field, value, ui_type):\n    ui_type = str(ui_type or "").upper()\n    if ui_type == "BOOLEAN":\n        return bool(value)\n    if ui_type == "INT":\n        if field == "seconds":\n            return max(1, int(math.ceil(float(value))))\n        return int(round(float(value)))\n    if ui_type == "FLOAT":\n        return float(value)\n    if ui_type in {"STRING", "COMBO"}:\n        return str(value)\n    return value\n\n\ndef _set_exact_widget(owner, node, field, value, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list):\n        return\n    for inp in node.get("inputs") or []:\n        if not isinstance(inp, dict):\n            continue\n        label = _norm_label(inp.get("label") or inp.get("name"))\n        if not _matches_field_label(field, label):\n            continue\n        idx = _widget_index(node, inp.get("name"))\n        if idx is None or idx >= len(values):\n            continue\n        # User-requested controls can be linked to helper nodes in a UI workflow\n        # (the current GGUF workflow links width/height to ResolutionSelector).\n        # Disconnect that incoming UI link so the patched widget value is the\n        # value that survives UI -> API conversion.\n        if field in {\n            "prompt", "width", "height", "seconds", "fps", "seed",\n            "image_strength", "text_to_video", "use_image", "use_audio",\n            "prompt_enhance", "tile_size", "tile_overlap",\n            "temporal_size", "temporal_overlap",\n        } and inp.get("link") is not None:\n            link_id = inp.get("link")\n            _remove_ui_link(owner, link_id)\n            inp["link"] = None\n            stats[f"disconnected_{field}"] = stats.get(f"disconnected_{field}", 0) + 1\n        values[idx] = _coerce_widget_value(field, value, inp.get("type"))\n        stats[field] = stats.get(field, 0) + 1\n\n\ndef _patch_primitives(node, values_by_field, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list) or not values:\n        return\n    node_type = str(node.get("type") or "")\n    title = _norm_label(node.get("title"))\n    if node_type in {"PrimitiveString", "PrimitiveStringMultiline"} and _matches_field_label("prompt", title):\n        values[0] = values_by_field["prompt"]\n        stats["prompt"] = stats.get("prompt", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["seconds"]:\n        values[0] = int(math.ceil(values_by_field["seconds"])) if node_type == "PrimitiveInt" else float(values_by_field["seconds"])\n        stats["seconds"] = stats.get("seconds", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["fps"]:\n        values[0] = int(values_by_field["fps"]) if node_type == "PrimitiveInt" else float(values_by_field["fps"])\n        stats["fps"] = stats.get("fps", 0) + 1\n    elif node_type == "PrimitiveInt" and title in ALIASES["seed"]:\n        values[0] = int(values_by_field["seed"])\n        stats["seed"] = stats.get("seed", 0) + 1\n\n\ndef patch_workflow(\n    ui, *, mode, prompt, image_name, last_image_name=None, audio_name, model_file,\n    width, height, seconds, fps, seed, image_strength,\n    tile_size, tile_overlap, temporal_size, temporal_overlap,\n):\n    text_mode = mode in {"t2v", "a2v"}\n    use_image = mode in {"i2v", "ia2v", "flf2v"}\n    use_audio = mode in {"a2v", "ia2v"}\n    values_by_field = {\n        "prompt": prompt, "width": width, "height": height, "seconds": seconds,\n        "fps": fps, "seed": seed, "image_strength": image_strength,\n        "video_vae": VIDEO_VAE, "audio_vae": AUDIO_VAE,\n        "text_encoder": TEXT_ENCODER, "upscaler": UPSCALER,\n        "text_to_video": text_mode, "use_image": use_image, "use_audio": use_audio,\n        "prompt_enhance": False, "tile_size": tile_size, "tile_overlap": tile_overlap,\n        "temporal_size": temporal_size, "temporal_overlap": temporal_overlap,\n        "model": model_file,\n    }\n    stats = {}\n\n    for owner, node in _iter_all_nodes(ui):\n        node_type = str(node.get("type") or "")\n        widgets = node.get("widgets_values")\n        _patch_primitives(node, values_by_field, stats)\n\n        if node_type == "LoadImage" and isinstance(widgets, list) and widgets:\n            # FLF2V assigns its ending image by the titled LoadImage node.\n            # FLF2V 依照 LoadImage 標題指定結尾圖片。\n            if mode == "flf2v" and "last" in _norm_label(node.get("title")):\n                widgets[0] = last_image_name\n                stats["last_image"] = stats.get("last_image", 0) + 1\n            else:\n                widgets[0] = image_name\n                stats["first_image"] = stats.get("first_image", 0) + 1\n            stats["load_image"] = stats.get("load_image", 0) + 1\n        elif node_type == "LoadAudio" and isinstance(widgets, list) and widgets:\n            widgets[0] = audio_name\n            stats["load_audio"] = stats.get("load_audio", 0) + 1\n        elif node_type in {"UnetLoaderGGUF", "UNETLoader"} and isinstance(widgets, list) and widgets:\n            widgets[0] = model_file\n            stats["model"] = stats.get("model", 0) + 1\n        elif node_type == "LatentUpscaleModelLoader" and isinstance(widgets, list) and widgets:\n            widgets[0] = UPSCALER\n            stats["upscaler"] = stats.get("upscaler", 0) + 1\n\n        for field, value in values_by_field.items():\n            _set_exact_widget(owner, node, field, value, stats)\n\n    # Contract checks: fail here rather than silently generating with stale defaults.\n    required = ["prompt", "width", "height", "seconds", "fps", "seed", "model"]\n    if mode in {"i2v", "ia2v", "flf2v"}:\n        required.append("load_image")\n    if mode == "flf2v":\n        required += ["first_image", "last_image"]\n    missing = [name for name in required if stats.get(name, 0) == 0]\n    if missing:\n        raise RuntimeError(\n            f"Workflow patch contract failed for mode={mode}: missing controls/nodes {missing}. "\n            f"Patch stats={stats}"\n        )\n    return ui, stats\n\n\n_OBJECT_INFO_CACHE = {"at": 0.0, "data": None}\n\n\ndef get_object_info_all(max_age=60):\n    now = time.time()\n    if _OBJECT_INFO_CACHE["data"] is None or now - _OBJECT_INFO_CACHE["at"] > max_age:\n        _OBJECT_INFO_CACHE["data"] = comfy_get("/object_info", timeout=60).json()\n        _OBJECT_INFO_CACHE["at"] = now\n    return _OBJECT_INFO_CACHE["data"]\n\n\ndef convert_workflow(ui_workflow):\n    response = comfy_post("/workflow/convert", json=ui_workflow, timeout=120)\n    payload = response.json()\n    if not isinstance(payload, dict) or not any(\n        isinstance(v, dict) and "class_type" in v for v in payload.values()\n    ):\n        raise RuntimeError(f"Workflow converter returned unexpected payload: {payload}")\n    return payload\n\n\ndef normalize_nested_random_noise(api_workflow):\n    """Use local Gaussian noise that preserves nested video/audio structure."""\n    converted = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "RandomNoise":\n            node["class_type"] = "LTX25NestedRandomNoise"\n            converted += 1\n    if converted == 0:\n        raise RuntimeError("Converted workflow contains no RandomNoise nodes to normalize.")\n    return converted\n\n\ndef normalize_api_workflow_for_gguf(api_workflow, model_file):\n    # Place GGUF diffusion on GPU 0 and encoders/VAEs on GPU 1.\n    # 將 GGUF Diffusion 指定至 GPU 0，文字編碼器與 VAE 指定至 GPU 1。\n    counts = {\'transformer\': 0, \'text_encoder\': 0, \'vae\': 0, \'upscaler\': 0}\n    for node in api_workflow.values():\n        kind = node.get(\'class_type\')\n        inputs = node.get(\'inputs\', {})\n        if kind in {\'UNETLoader\', \'UnetLoaderGGUF\'}:\n            node[\'class_type\'] = \'LTX25GGUFLoaderGPU\'\n            node[\'inputs\'] = {\'unet_name\': model_file, \'device\': \'cuda:0\'}\n            counts[\'transformer\'] += 1\n        elif kind == \'CLIPLoader\':\n            node[\'class_type\'] = \'LTX25CLIPLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'text_encoder\'] += 1\n        elif kind == \'VAELoader\':\n            node[\'class_type\'] = \'LTX25VAELoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'vae\'] += 1\n        elif kind == \'LatentUpscaleModelLoader\':\n            node[\'class_type\'] = \'LTX25UpscalerLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'upscaler\'] += 1\n    # Reject a workflow that accidentally bypasses either GPU or the two-stage upscaler.\n    # 若 Workflow 略過任一 GPU 或缺少雙階段 Upscaler，直接拒絕執行。\n    if not all(counts.values()):\n        raise RuntimeError(f\'Missing dual-GPU component loaders in converted workflow: {counts}\')\n    print(\'Independent GPU loader mapping:\', counts)\n    return sum(counts.values())\n\n\ndef inject_frozen_external_audio(api_workflow, audio_name, seconds):\n    """Inject existing audio into both stages of the distilled AV workflow.\n\n    Local-only path:\n      LoadAudio -> TrimAudioDuration -> VAEEncodeAudio\n      -> LTXVSetAudioRefTokens -> frozen_audio\n      -> every active LTXVConcatAVLatent audio input\n\n    The normal positive/negative conditioning graph is left unchanged; only the\n    node\'s frozen_audio output is used. For audio-driven modes, both samplers are\n    switched to plain Euler.\n    """\n    if not audio_name:\n        raise RuntimeError("External audio injection requires audio_name.")\n\n    audio_vae_loader = None\n    conditioning_node = None\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "VAELoader" and (node.get("inputs") or {}).get("vae_name") == AUDIO_VAE:\n            audio_vae_loader = str(node_id)\n        if conditioning_node is None and node.get("class_type") == "LTXVConditioning":\n            conditioning_node = str(node_id)\n\n    if audio_vae_loader is None:\n        raise RuntimeError(f"Audio VAELoader not found for {AUDIO_VAE!r}.")\n    if conditioning_node is None:\n        raise RuntimeError("LTXVConditioning not found in converted workflow.")\n\n    def fresh_id(base):\n        candidate = base\n        n = 1\n        while candidate in api_workflow:\n            n += 1\n            candidate = f"{base}_{n}"\n        return candidate\n\n    load_id = fresh_id("ltx25_external_audio")\n    trim_id = fresh_id("ltx25_trim_audio")\n    encode_id = fresh_id("ltx25_encode_audio")\n    freeze_id = fresh_id("ltx25_freeze_audio")\n\n    api_workflow[load_id] = {\n        "class_type": "LoadAudio",\n        "inputs": {"audio": audio_name},\n        "_meta": {"title": "LTX25 External Audio"},\n    }\n    api_workflow[trim_id] = {\n        "class_type": "TrimAudioDuration",\n        "inputs": {\n            "audio": [load_id, 0],\n            "start_index": 0.0,\n            "duration": float(seconds),\n        },\n        "_meta": {"title": "LTX25 Trim External Audio"},\n    }\n    api_workflow[encode_id] = {\n        "class_type": "VAEEncodeAudio",\n        "inputs": {\n            "audio": [trim_id, 0],\n            "vae": [audio_vae_loader, 0],\n        },\n        "_meta": {"title": "LTX25 Encode External Audio"},\n    }\n    api_workflow[freeze_id] = {\n        "class_type": "LTXVSetAudioRefTokens",\n        "inputs": {\n            "positive": [conditioning_node, 0],\n            "negative": [conditioning_node, 1],\n            "audio_latent": [encode_id, 0],\n        },\n        "_meta": {"title": "LTX25 Freeze External Audio"},\n    }\n\n    concat_count = 0\n    sampler_count = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        class_type = node.get("class_type")\n        inputs = node.get("inputs")\n        if not isinstance(inputs, dict):\n            continue\n        if class_type == "LTXVConcatAVLatent":\n            # Official distilled A2V keeps the same encoded audio frozen in both stages.\n            inputs["audio_latent"] = [freeze_id, 2]\n            concat_count += 1\n        elif class_type == "KSamplerSelect":\n            inputs["sampler_name"] = "euler"\n            sampler_count += 1\n\n    if concat_count < 2:\n        raise RuntimeError(\n            "Expected a two-stage AV workflow with at least two LTXVConcatAVLatent nodes; "\n            f"found {concat_count}. The pinned workflow may have changed."\n        )\n    if sampler_count < 2:\n        raise RuntimeError(\n            "Expected at least two KSamplerSelect nodes for the two-stage audio path; "\n            f"found {sampler_count}."\n        )\n\n    return {\n        "frozen_audio_concat_nodes": concat_count,\n        "euler_samplers": sampler_count,\n    }\n\n\ndef _is_connection(value):\n    return (\n        isinstance(value, list) and len(value) == 2\n        and isinstance(value[0], (str, int)) and isinstance(value[1], int)\n    )\n\n\ndef validate_api_workflow(api_workflow):\n    """Structural validation against ComfyUI /object_info without queueing GPU work.\\n\\n    Supports normal required inputs plus newer grouped/dynamic inputs\\n    (for example Autogrow values.a / values.b / ...).\\n    """\n    object_info = get_object_info_all()\n    errors = []\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict) or "class_type" not in node:\n            continue\n        class_type = node["class_type"]\n        spec = object_info.get(class_type)\n        if not isinstance(spec, dict):\n            errors.append(f"node {node_id}: class_type {class_type!r} is not registered")\n            continue\n        inputs = node.get("inputs") or {}\n        groups = spec.get("input") or {}\n        required = groups.get("required") or {}\n        optional = groups.get("optional") or {}\n        for name in required:\n            if name in inputs:\n                continue\n\n            # Switch branches are lazy/conditional inputs. Only the branch\n            # selected by `switch` is requested by current ComfyUI.\n            if class_type in {"ComfySwitchNode", "ComfySoftSwitchNode"} and name in {"on_false", "on_true"}:\n                switch_value = inputs.get("switch")\n                if isinstance(switch_value, bool):\n                    selected = "on_true" if switch_value else "on_false"\n                    if name != selected:\n                        continue\n                    # If the statically selected branch is missing, fall through\n                    # and report it. A linked/dynamic switch is deferred.\n                else:\n                    continue\n\n            # ComfyUI\'s newer schema supports grouped/dynamic inputs such as\n            # io.Autogrow.Input("values", ...). The converted API workflow does\n            # not contain a literal "values" key; it contains children such as\n            # "values.a", "values.b", ... instead.\n            #\n            # Treat the required group as present when at least one child key\n            # exists. This is generic and is not hard-coded to ComfyMathExpression.\n            dynamic_prefix = f"{name}."\n            has_dynamic_child = any(\n                isinstance(input_name, str) and input_name.startswith(dynamic_prefix)\n                for input_name in inputs\n            )\n            if has_dynamic_child:\n                continue\n\n            errors.append(f"node {node_id} {class_type}: missing required input {name!r}")\n\n        definitions = dict(optional)\n        definitions.update(required)\n        for name, value in inputs.items():\n            # LoadImage/LoadAudio file lists are dynamic. /object_info may have\n            # been cached before a request uploaded its file, so do not use the\n            # cached combo list as our own file-existence validator. ComfyUI\'s\n            # node validation remains authoritative when /prompt is queued.\n            if (class_type == "LoadAudio" and name == "audio") or (class_type == "LoadImage" and name == "image"):\n                continue\n\n            definition = definitions.get(name)\n            if not isinstance(definition, (list, tuple)) or not definition:\n                continue\n            choices = definition[0]\n            if isinstance(choices, list) and not _is_connection(value):\n                if value not in choices:\n                    preview = choices[:20]\n                    errors.append(\n                        f"node {node_id} {class_type}.{name}: {value!r} not in selectable choices {preview!r}"\n                    )\n    if errors:\n        raise RuntimeError("ComfyUI structural preflight failed:\\n" + "\\n".join(errors[:50]))\n    return True\n\n\ndef queue_prompt(api_workflow):\n    response = comfy_post(\n        "/prompt",\n        json={"prompt": api_workflow, "client_id": str(uuid.uuid4())},\n        timeout=120,\n    )\n    payload = response.json()\n    if payload.get("node_errors"):\n        raise RuntimeError(\n            "ComfyUI rejected workflow:\\n"\n            + json.dumps(payload["node_errors"], ensure_ascii=False, indent=2)\n        )\n    prompt_id = payload.get("prompt_id")\n    if not prompt_id:\n        raise RuntimeError(f"ComfyUI did not return prompt_id: {payload}")\n    return prompt_id\n\n\ndef comfy_crash_diagnostics():\n    parts = []\n\n    pid = None\n    try:\n        if COMFY_PID_PATH.is_file():\n            pid = int(COMFY_PID_PATH.read_text(encoding="utf-8").strip())\n            try:\n                os.kill(pid, 0)\n                parts.append(f"ComfyUI PID {pid} still exists.")\n            except OSError:\n                parts.append(f"ComfyUI PID {pid} is no longer running.")\n    except Exception as exc:\n        parts.append(f"PID check failed: {exc}")\n\n    try:\n        meminfo = {}\n        for line in Path("/proc/meminfo").read_text().splitlines():\n            if ":" in line:\n                key, value = line.split(":", 1)\n                meminfo[key] = value.strip()\n        parts.append(\n            "Host memory after failure: "\n            f"MemAvailable={meminfo.get(\'MemAvailable\', \'?\')}, "\n            f"SwapFree={meminfo.get(\'SwapFree\', \'?\')}."\n        )\n    except Exception:\n        pass\n\n    try:\n        gpu = subprocess.run(\n            [\n                "nvidia-smi",\n                "--query-gpu=memory.total,memory.used,memory.free",\n                "--format=csv,noheader,nounits",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=15,\n        )\n        if gpu.stdout.strip():\n            parts.append("GPU MiB total,used,free: " + gpu.stdout.strip())\n    except Exception:\n        pass\n\n    try:\n        if COMFY_LOG_PATH.is_file():\n            tail = COMFY_LOG_PATH.read_text(errors="replace")[-7000:]\n            parts.append("ComfyUI log tail:\\n" + tail)\n    except Exception as exc:\n        parts.append(f"Could not read ComfyUI log: {exc}")\n\n    return "\\n".join(parts)\n\n\ndef _looks_like_connection_loss(exc):\n    text = str(exc).lower()\n    return (\n        "connection refused" in text\n        or "failed to establish a new connection" in text\n        or "max retries exceeded" in text\n    )\n\n\ndef wait_prompt(prompt_id, timeout=3600):\n    deadline = time.time() + timeout\n    while time.time() < deadline:\n        try:\n            payload = comfy_get(\n                f"/history/{prompt_id}",\n                timeout=60,\n            ).json()\n        except Exception as exc:\n            if _looks_like_connection_loss(exc):\n                raise RuntimeError(\n                    "ComfyUI became unreachable during generation. "\n                    "The ComfyUI process probably exited or was killed by memory pressure.\\n"\n                    + comfy_crash_diagnostics()\n                ) from exc\n            raise\n\n        if prompt_id in payload:\n            history = payload[prompt_id]\n            status = history.get("status", {})\n            if status.get("status_str") == "error":\n                # Show only the failing node and traceback, never megabytes of latent tensors.\n                # 只輸出失敗節點與 traceback，避免將大型 latent tensor 列印到畫面。\n                details = [entry[1] for entry in status.get("messages", []) if entry[0] == "execution_error"]\n                if details:\n                    error = details[-1]\n                    raise RuntimeError(\n                        f"ComfyUI node {error.get(\'node_id\')} ({error.get(\'node_type\')}): "\n                        f"{error.get(\'exception_type\')}: {error.get(\'exception_message\')}\\n"\n                        + "".join(error.get(\'traceback\', [])[-12:])\n                    )\n                raise RuntimeError(f"ComfyUI failed: {status.get(\'messages\', [])[-3:]}")\n            return history\n        time.sleep(3)\n    raise TimeoutError(f"Timed out waiting for ComfyUI prompt {prompt_id}")\n\n\ndef extract_output_item(history):\n    # Prefer video-like outputs. Some nodes use "videos", older wrappers use\n    # "gifs", and newer/custom nodes may expose generic "files".\n    video_exts = {".mp4", ".webm", ".mov", ".mkv", ".gif"}\n    fallback = None\n\n    for output in history.get("outputs", {}).values():\n        if not isinstance(output, dict):\n            continue\n        for key in ("videos", "gifs", "images", "files"):\n            items = output.get(key, [])\n            if not isinstance(items, list):\n                continue\n            for item in items:\n                if not isinstance(item, dict) or not item.get("filename"):\n                    continue\n                suffix = Path(item["filename"]).suffix.lower()\n                if key in {"videos", "gifs"} or suffix in video_exts:\n                    return item\n                if fallback is None:\n                    fallback = item\n\n    if fallback is not None:\n        return fallback\n    raise RuntimeError("Generation completed but no output file was found in ComfyUI history.")\n\n\ndef materialize_output(job_id, item):\n    output_dir = Path(os.environ.get("LTX25_OUTPUT_DIR", "/kaggle/working/ltx25_api_outputs"))\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    filename = item.get("filename", f"{job_id}.mp4")\n    suffix = Path(filename).suffix or ".mp4"\n    target = output_dir / f"{job_id}{suffix}"\n\n    response = comfy_get(\n        "/view",\n        params=item,\n        timeout=600,\n    )\n    target.write_bytes(response.content)\n    return target\n\n\ndef mux_original_audio(job_id, video_path, audio_name):\n    """Mux the original uploaded audio back onto the generated video.\n\n    The latent audio drives generation, but the delivered MP4 uses the user\'s\n    original waveform rather than a VAE-decoded reconstruction.\n    """\n    audio_path = COMFY_DIR / "input" / audio_name\n    if not audio_path.is_file():\n        raise RuntimeError(f"Uploaded audio disappeared: {audio_path}")\n\n    output_dir = Path(os.environ.get("LTX25_OUTPUT_DIR", "/kaggle/working/ltx25_api_outputs"))\n    target = output_dir / f"{job_id}_with_input_audio.mp4"\n    cmd = [\n        "ffmpeg", "-y", "-v", "error",\n        "-i", str(video_path),\n        "-i", str(audio_path),\n        "-map", "0:v:0",\n        "-map", "1:a:0",\n        "-c:v", "copy",\n        "-c:a", "aac",\n        "-b:a", "192k",\n        "-shortest",\n        str(target),\n    ]\n    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)\n    if result.returncode != 0:\n        raise RuntimeError(f"ffmpeg audio mux failed: {result.stderr[:4000]}")\n    return target\n\n\ndef looks_like_oom(exc):\n    text = str(exc).lower()\n    needles = [\n        "cuda out of memory",\n        "out of memory",\n        "not enough memory",\n        "failed to allocate",\n        "cublas_status_alloc_failed",\n        "cudaerror: out of memory",\n    ]\n    return any(n in text for n in needles)\n\n\ndef choose_workflow(mode):\n    if mode == "flf2v":\n        if not FLF2V_WORKFLOW.is_file():\n            raise RuntimeError(f"FLF2V workflow missing: {FLF2V_WORKFLOW}")\n        return FLF2V_WORKFLOW\n    if mode not in {"t2v", "i2v", "a2v", "ia2v"}:\n        raise RuntimeError(f"Unsupported mode: {mode}")\n    if not T2V_I2V_WORKFLOW.is_file():\n        raise RuntimeError(f"Base workflow missing: {T2V_I2V_WORKFLOW}")\n    return T2V_I2V_WORKFLOW\n\n\ndef prepare_api_workflow(params, tier):\n    mode = params["mode"]\n    model_file = MODEL_FILES[tier]\n    workflow_path = choose_workflow(mode)\n    ui = json.loads(workflow_path.read_text(encoding="utf-8"))\n    ui, patch_stats = patch_workflow(\n        ui,\n        mode=mode,\n        prompt=params["prompt"],\n        image_name=params["image_name"],\n        last_image_name=params.get("last_image_name"),\n        audio_name=params["audio_name"],\n        model_file=model_file,\n        width=params["width"],\n        height=params["height"],\n        seconds=params["seconds"],\n        fps=params["fps"],\n        seed=params["seed"],\n        image_strength=params["image_strength"],\n        tile_size=params["vae_tile_size"],\n        tile_overlap=params["vae_tile_overlap"],\n        temporal_size=params["vae_temporal_size"],\n        temporal_overlap=params["vae_temporal_overlap"],\n    )\n    api_workflow = convert_workflow(ui)\n    nested_noise_nodes = normalize_nested_random_noise(api_workflow)\n    patch_stats["nested_random_noise"] = nested_noise_nodes\n    normalized_loaders = normalize_api_workflow_for_gguf(api_workflow, model_file)\n    if mode == "flf2v":\n        # Both stages must condition on the opening and closing frames.\n        # 兩個階段都必須加入起始與結尾影格條件。\n        guides = [node for node in api_workflow.values() if node.get("class_type") == "LTXVAddGuide"]\n        frame_ids = [node.get("inputs", {}).get("frame_idx") for node in guides]\n        if frame_ids.count(0) < 2 or frame_ids.count(-1) < 2:\n            raise RuntimeError(f"FLF2V requires first/last guides in both stages: {frame_ids}")\n        image_names = {node.get("inputs", {}).get("image") for node in api_workflow.values()\n                       if node.get("class_type") == "LoadImage"}\n        if not {params["image_name"], params["last_image_name"]}.issubset(image_names):\n            raise RuntimeError(f"FLF2V did not bind both images: {image_names}")\n    if normalized_loaders:\n        patch_stats["api_unet_to_gguf"] = normalized_loaders\n    if mode in {"a2v", "ia2v"}:\n        audio_stats = inject_frozen_external_audio(\n            api_workflow, params["audio_name"], params["seconds"]\n        )\n        patch_stats.update(audio_stats)\n    validate_api_workflow(api_workflow)\n    return workflow_path, api_workflow, patch_stats\n\n\ndef run_once(job_id, params, tier):\n    model_file = MODEL_FILES[tier]\n\n    loader_ready, gguf_choices, _ = get_gguf_loader_choices()\n    if not loader_ready:\n        raise RuntimeError("ComfyUI node UnetLoaderGGUF is not registered.")\n    if model_file not in gguf_choices:\n        raise RuntimeError(\n            f"Model exists on disk but UnetLoaderGGUF cannot select {model_file!r}. "\n            f"Choices={gguf_choices}"\n        )\n\n    job_update(job_id, status="preparing", message=f"Patching/validating workflow with {tier}.")\n    workflow_path, api_workflow, patch_stats = prepare_api_workflow(params, tier)\n    job_update(\n        job_id,\n        workflow=str(workflow_path),\n        patch_stats=patch_stats,\n        message=f"Workflow structurally validated with {tier}; queueing ComfyUI.",\n    )\n\n    if not KEEP_MODELS_RESIDENT:\n        free_comfy()\n        time.sleep(1)\n    prompt_id = queue_prompt(api_workflow)\n    job_update(\n        job_id,\n        prompt_id=prompt_id,\n        status="running",\n        message=f"ComfyUI is generating with {tier}.",\n    )\n\n    history = wait_prompt(prompt_id)\n    item = extract_output_item(history)\n    result_path = materialize_output(job_id, item)\n    if params["mode"] in {"a2v", "ia2v"}:\n        result_path = mux_original_audio(job_id, result_path, params["audio_name"])\n    if not KEEP_MODELS_RESIDENT:\n        free_comfy()\n    return result_path\n\n\ndef worker(job_id, params):\n    try:\n        job_update(job_id, status="preparing", message="Preparing local ComfyUI workflow.")\n        # Model policy comes from this Kaggle notebook.\n        configured_chain = [PRIMARY_MODEL_TIER] + [\n            tier for tier in FALLBACK_MODEL_TIERS\n            if tier != PRIMARY_MODEL_TIER\n        ]\n\n        for tier in configured_chain:\n            if tier not in MODEL_FILES:\n                raise RuntimeError(\n                    f"Notebook model policy contains unsupported tier {tier!r}. "\n                    f"Supported: {sorted(MODEL_FILES)}"\n                )\n\n        # Skip configured fallback tiers that have not been downloaded.\n        available_chain = []\n        for tier in configured_chain:\n            model_path = COMFY_DIR / "models/diffusion_models" / MODEL_FILES[tier]\n            if model_path.is_file():\n                available_chain.append(tier)\n\n        if not available_chain:\n            raise RuntimeError(\n                "No configured model tier is downloaded. "\n                f"Notebook policy: {configured_chain}"\n            )\n\n        result = None\n        last_error = None\n        for index, tier in enumerate(available_chain):\n            try:\n                result = run_once(job_id, params, tier)\n                break\n            except Exception as exc:\n                last_error = exc\n                is_last = index == len(available_chain) - 1\n                if is_last or not looks_like_oom(exc):\n                    raise\n\n                next_tier = available_chain[index + 1]\n                free_comfy()\n                job_update(\n                    job_id,\n                    status="retrying",\n                    active_model_tier=next_tier,\n                    message=f"{tier} failed with an OOM-like error; retrying {next_tier}.",\n                    last_tier_error=str(exc),\n                )\n\n        if result is None:\n            raise RuntimeError(f"All model tiers failed. Last error: {last_error}")\n\n        job_update(\n            job_id,\n            status="completed",\n            result_path=str(result),\n            message="Generation completed.",\n        )\n\n    except Exception as exc:\n        if not KEEP_MODELS_RESIDENT:\n            free_comfy()\n        job_update(\n            job_id,\n            status="failed",\n            error=str(exc),\n            message="Generation failed.",\n        )\n\n\ndef _sample_preflight_params(mode):\n    return {\n        "mode": mode,\n        "prompt": "LTX-2.5 structural preflight",\n        "width": 608,\n        "height": 352,\n        "seconds": 3.0,\n        "fps": 24,\n        "seed": 42,\n        "image_strength": 0.7,\n        "vae_tile_size": VAE_TILE_SIZE,\n        "vae_tile_overlap": VAE_TILE_OVERLAP,\n        "vae_temporal_size": VAE_TEMPORAL_SIZE,\n        "vae_temporal_overlap": VAE_TEMPORAL_OVERLAP,\n        "image_name": ensure_placeholder_image(),\n        "audio_name": ensure_placeholder_audio(),\n        "audio_info": None,\n    }\n\n\n_PREFLIGHT_CACHE = {"at": 0.0, "data": None}\n\n\ndef preflight_all_modes(max_age=30):\n    now = time.time()\n    if _PREFLIGHT_CACHE["data"] is not None and now - _PREFLIGHT_CACHE["at"] <= max_age:\n        return _PREFLIGHT_CACHE["data"]\n    result = {}\n    for mode in ("t2v", "i2v", "a2v", "ia2v"):\n        try:\n            workflow_path, _, stats = prepare_api_workflow(\n                _sample_preflight_params(mode), PRIMARY_MODEL_TIER\n            )\n            result[mode] = {\n                "ok": True,\n                "workflow": str(workflow_path),\n                "patch_stats": stats,\n            }\n        except Exception as exc:\n            result[mode] = {\n                "ok": False,\n                "workflow": str(T2V_I2V_WORKFLOW),\n                "error": str(exc),\n            }\n    _PREFLIGHT_CACHE["at"] = now\n    _PREFLIGHT_CACHE["data"] = result\n    return result\n\n\n'
ENGINE_PATH = ROOT / "ltx25_kaggle_engine.py"
ENGINE_PATH.write_text(ENGINE_CODE, encoding="utf-8")
print("Original workflow engine + dual-GPU loader patch ready.")
LTX25_NOISE_DIR = COMFY_DIR / "custom_nodes/LTX25LocalNoise"
LTX25_NOISE_DIR.mkdir(parents=True, exist_ok=True)
LTX25_NOISE_CODE = 'import numpy as np\nimport torch\nimport comfy.nested_tensor\n\n\ndef _noise_for_tensor(tensor, generator, noise_inds=None):\n    if noise_inds is None:\n        return torch.randn(\n            tensor.size(), dtype=torch.float32, layout=tensor.layout,\n            generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n\n    unique_inds, inverse = np.unique(noise_inds, return_inverse=True)\n    noises = []\n    for i in range(unique_inds[-1] + 1):\n        noise = torch.randn(\n            [1] + list(tensor.size())[1:], dtype=torch.float32,\n            layout=tensor.layout, generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n        if i in unique_inds:\n            noises.append(noise)\n    noises = [noises[i] for i in inverse]\n    return torch.cat(noises, axis=0)\n\n\nclass _LTX25NestedNoise:\n    def __init__(self, seed):\n        self.seed = int(seed)\n\n    def generate_noise(self, input_latent):\n        latent = input_latent["samples"]\n        noise_inds = input_latent.get("batch_index")\n        generator = torch.manual_seed(self.seed)\n\n        if getattr(latent, "is_nested", False):\n            parts = [\n                _noise_for_tensor(part, generator, noise_inds)\n                for part in latent.unbind()\n            ]\n            return comfy.nested_tensor.NestedTensor(parts)\n\n        return _noise_for_tensor(latent, generator, noise_inds)\n\n\nclass LTX25NestedRandomNoise:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "noise_seed": (\n                    "INT",\n                    {\n                        "default": 0,\n                        "min": 0,\n                        "max": 0xFFFFFFFFFFFFFFFF,\n                        "control_after_generate": True,\n                    },\n                )\n            }\n        }\n\n    RETURN_TYPES = ("NOISE",)\n    FUNCTION = "get_noise"\n    CATEGORY = "LTX25/local"\n\n    def get_noise(self, noise_seed):\n        return (_LTX25NestedNoise(noise_seed),)\n\n\nNODE_CLASS_MAPPINGS = {"LTX25NestedRandomNoise": LTX25NestedRandomNoise}\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "LTX25NestedRandomNoise": "LTX25 Nested Random Noise"\n}\n'
(LTX25_NOISE_DIR / "__init__.py").write_text(LTX25_NOISE_CODE, encoding="utf-8")

'''
Install a small standalone device-placement node pack, not the archived package.
安裝獨立的 GPU 裝置選擇節點，不使用封存的套件。
'''
DUAL_NODE_DIR = COMFY_DIR / 'custom_nodes/zz_ltx25_gpu_components'
DUAL_NODE_DIR.mkdir(parents=True, exist_ok=True)
DUAL_NODE_SOURCE = '# Select independent GPUs without archived DisTorch2 or model sharding.\n# 不使用封存的 DisTorch2，以元件方式分配兩張 GPU。\nfrom contextlib import nullcontext\nfrom contextvars import ContextVar\n\n# Access ComfyUI\'s existing model-loading functions.\n# 使用 ComfyUI 原有的模型載入方法。\nimport comfy.model_management as mm\n# Access the original ComfyUI tensor-loading function.\n# 取得 ComfyUI 原本的張量載入函式。\nimport comfy.memory_management as memory_management\n\n# Preserve the original reader for every other model.\n# 其他模型仍沿用原本的讀取方式。\n_original_read_tensor = memory_management.read_tensor_file_slice_into\n\n# Fall back to PyTorch copying for the LTX spatial upscaler only.\n# 僅讓 LTX Spatial Upscaler 改用 PyTorch 複製，避開 AIMDO 傳輸錯誤。\ndef _ltx25_read_tensor(tensor, destination, stream=None, destination2=None):\n    # Locate the checkpoint associated with this tensor, if available.\n    # 取得此張量所屬的權重檔案路徑。\n    tensor_storage = tensor.untyped_storage() if hasattr(tensor, \'untyped_storage\') else None\n    # Check the source metadata attached to an mmap-backed tensor.\n    # 讀取 mmap 張量所附帶的來源資訊。\n    source_path = getattr(tensor_storage, \'_comfy_source_path\', None)\n    # Use the original file handle when source metadata is absent.\n    # 若沒有來源標記，改從原始檔案控制代碼判斷。\n    if source_path is None:\n        # Read the file-slice descriptor used by AIMDO.\n        # 讀取 AIMDO 使用的檔案區段資訊。\n        file_slice = getattr(tensor_storage, \'_comfy_tensor_file_slice\', None)\n        # Derive a filename only when file-slice metadata exists.\n        # 只有在存在檔案區段資訊時，才取得檔案名稱。\n        source_path = getattr(getattr(file_slice, \'file_ref\', None), \'name\', \'\')\n    # Redirect only the known failing LTX-2.5 upscaler weights.\n    # 只對先前出錯的 LTX-2.5 Upscaler 權重採用替代路徑。\n    if \'ltx-2.5-latent-spatial-upscaler\' in str(source_path).lower():\n        # Report the bypass only on its first actual use.\n        # 只在第一次確實使用替代載入路徑時顯示訊息。\n        if not getattr(_ltx25_read_tensor, \'_reported\', False):\n            # Leave one diagnostic line in the ComfyUI log.\n            # 在 ComfyUI Log 保留一行診斷訊息。\n            print(\'[LTX25] Spatial Upscaler: bypassing AIMDO file-to-device transfer\')\n            # Suppress repeated messages for later weight tensors.\n            # 後續張量不再重複印出訊息。\n            _ltx25_read_tensor._reported = True\n        # Let ComfyUI\'s native tensor copy handle the mmap-backed weight.\n        # 交由 ComfyUI 後續的 PyTorch 張量複製流程處理 mmap 權重。\n        return False\n    # Keep AIMDO\'s efficient path for all other models.\n    # 其他模型保留 AIMDO 的低 RAM 載入路徑。\n    return _original_read_tensor(tensor, destination, stream=stream, destination2=destination2)\n\n# Register the targeted fallback before any model is loaded.\n# 在載入任何模型之前，註冊這項局部替代機制。\nmemory_management.read_tensor_file_slice_into = _ltx25_read_tensor\n\nimport folder_paths\nimport torch\nfrom nodes import NODE_CLASS_MAPPINGS as CORE_NODES\n\n# Keep device selection local to a single loader execution.\n# 讓裝置選擇只影響目前執行的 Loader，不改變其他節點的預設行為。\n_active_device = ContextVar(\'ltx25_device\', default=None)\n_original_torch_device = mm.get_torch_device\n_original_text_device = mm.text_encoder_device\n_original_vae_device = mm.vae_device\n\n# Return the selected device only when a loader requests one explicitly.\n# 只有 Loader 明確指定裝置時，才覆寫原本的裝置查詢。\ndef _selected_device(original):\n    selected = _active_device.get()\n    return torch.device(selected) if selected is not None else original()\n\n# Patch the three official device lookup functions without changing memory accounting.\n# 僅覆寫 ComfyUI 三個裝置查詢函式，不改寫記憶體管理程式。\nmm.get_torch_device = lambda: _selected_device(_original_torch_device)\nmm.text_encoder_device = lambda: _selected_device(_original_text_device)\nmm.vae_device = lambda: _selected_device(_original_vae_device)\n\n# Declare the common GPU choices used by every component loader.\n# 統一定義各模型元件使用的 GPU 選項。\nDEVICES = [\'cuda:0\', \'cuda:1\', \'cpu\']\n\n# Convert a new-style ComfyNode output into the legacy tuple format.\n# 將新版 ComfyNode 的 NodeOutput 轉成舊式節點可使用的 tuple。\ndef _as_tuple(result):\n    # Use the wrapped values directly when the node already returns a tuple.\n    # 若原本就回傳 tuple，直接使用，不額外轉換。\n    if isinstance(result, tuple):\n        return result\n    # Newer ComfyUI returns NodeOutput with its values stored in args.\n    # 新版 ComfyUI 的 NodeOutput 會將輸出值保存在 args。\n    if hasattr(result, \'args\'):\n        return tuple(result.args)\n    # Fail early rather than returning an invalid node payload.\n    # 如果輸出格式無法辨識，立即拋出錯誤，不讓壞資料流入 sampling。\n    raise TypeError(f\'Unsupported node output type: {type(result).__name__}\')\n\n# Locate an observable load_device field to verify the selected GPU.\n# 查找模型或 patcher 的 load_device，以確認實際裝置。\ndef _load_device(value):\n    # Check the public wrapper and common patcher locations.\n    # 檢查模型本體與常見的 patcher 欄位。\n    for obj in (value, getattr(value, \'patcher\', None), getattr(value, \'model_patcher\', None)):\n        # Skip containers that cannot report a model device.\n        # 忽略無法提供模型裝置的物件。\n        if obj is None:\n            continue\n        # Check explicit inference allocation before other placement fields.\n        # 優先確認模型實際指定的載入裝置。\n        found = getattr(obj, \'load_device\', None)\n        # Return the first explicit device from the patcher.\n        # 回傳第一個有效的載入裝置。\n        if found is not None:\n            return str(found)\n    # Report non-verifiable models rather than inventing a device.\n    # 若無法從物件確認裝置，不推測載入結果。\n    return None\n\n# Use either the legacy FUNCTION API or ComfyNode\'s execute method.\n# 同時支援舊式 FUNCTION 與新版 ComfyNode execute API。\ndef _load_component(name, device, **inputs):\n    # Find the actual core node instead of inheriting incompatible schemas.\n    # 直接取得原始節點，不繼承可能不相容的節點 Schema。\n    node = CORE_NODES.get(name)\n    # Reject missing plugins before sampling or GPU allocation.\n    # 若缺少必要外掛，先報錯，避免開始耗時生成。\n    if node is None:\n        raise RuntimeError(f\'Missing upstream ComfyUI node: {name}\')\n    # Set the target GPU only for this execution.\n    # 將 GPU 指定範圍限制在目前 Loader 的執行過程。\n    token = _active_device.set(device)\n    # Make kernels that use torch.cuda.current_device follow the same GPU.\n    # 讓依賴 torch.cuda.current_device 的 CUDA kernel 也指向正確 GPU。\n    context = torch.cuda.device(device) if device.startswith(\'cuda:\') else nullcontext()\n    try:\n        # Execute original loader code with the selected CUDA device active.\n        # 使用原本的 Loader 程式，同時設定對應 CUDA 裝置。\n        with context:\n            # Prefer the legacy function when the node exposes one.\n            # 如果節點提供舊式 FUNCTION，優先沿用原本載入方法。\n            if hasattr(node, \'FUNCTION\'):\n                # Invoke the original node without passing the custom GPU selector.\n                # 不將自訂 GPU 參數傳入原始 Loader，避免參數不相容。\n                result = getattr(node(), node.FUNCTION)(**inputs)\n            # Handle newer ComfyNode definitions that only provide execute.\n            # 若使用新版 ComfyNode API，改呼叫 execute。\n            elif hasattr(node, \'execute\'):\n                # Use the actual core ComfyNode class method.\n                # 呼叫核心節點提供的新版執行方法。\n                result = node.execute(**inputs)\n            else:\n                # Stop at an incompatible upstream API instead of guessing.\n                # 遇到無法辨識的新版 API 就直接拋出錯誤，避免錯誤分配 GPU。\n                raise RuntimeError(f\'Unsupported upstream loader API: {name}\')\n        # Normalize node output and verify its load device if observable.\n        # 整理節點輸出，並在可取得資訊時核對載入裝置。\n        values = _as_tuple(result)\n        # Reject empty output from any model loader.\n        # 拒絕沒有回傳模型的 Loader。\n        if not values:\n            raise RuntimeError(f\'{name} returned no model\')\n        # Compare actual model device with the requested assignment.\n        # 將模型的實際裝置與指定 GPU 比對。\n        actual = _load_device(values[0])\n        # Fail immediately if ComfyUI selected the wrong device.\n        # 若實際裝置與預期不同，立即中止並提示。\n        if actual is not None and actual != device:\n            raise RuntimeError(f\'{name}: requested {device}, actual load_device={actual}\')\n        # Distinguish observed placement from selection-only cases.\n        # 區分已驗證載入位置與僅完成指定裝置的情況。\n        print(f\'[LTX25 dualGPU] {name}: requested={device}, observed={actual or "not exposed"}\')\n        # Return the result in a format compatible with old-style custom nodes.\n        # 使用舊式自訂節點可接收的 tuple 格式回傳。\n        return values\n    finally:\n        # Restore original device behavior for subsequent graph nodes.\n        # 在 Loader 結束後恢復其他節點的預設裝置選擇。\n        _active_device.reset(token)\n\n# Use the core GGUF loader while explicitly selecting GPU 0.\n# 使用 ComfyUI-GGUF 原有 Loader，明確指定 GPU 0。\nclass LTX25GGUFLoaderGPU:\n    # Expose the same model choices that the installed GGUF plugin recognizes.\n    # 使用目前 GGUF 外掛提供的模型清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Declare exact API names required by the existing workflow.\n        # 定義原始 Workflow 需要的 GGUF 模型名稱欄位。\n        return {\'required\': {\'unet_name\': (folder_paths.get_filename_list(\'unet_gguf\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load a quantized LTX transformer through the upstream GGUF node.\n    # 透過 GGUF 外掛載入量化 LTX Transformer。\n    def load(self, unet_name, device):\n        # Reuse the GGUF implementation without any DisTorch2 modifications.\n        # 完全沿用 GGUF Loader，不加入 DisTorch2 的權重修補邏輯。\n        return _load_component(\'UnetLoaderGGUF\', device, unet_name=unet_name)\n\n# Use the standard CLIP loader with a separate GPU allocation.\n# 將 Gemma 文字編碼器分配到另一張 GPU。\nclass LTX25CLIPLoaderGPU:\n    # Keep the original CLIPLoader model-type options.\n    # 保留 ComfyUI 原本的 CLIPLoader 類型選項。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Extend the original choice list with the two CUDA devices.\n        # 在原本的文字模型選單加入雙 GPU 裝置選擇。\n        return {\'required\': {\'clip_name\': (folder_paths.get_filename_list(\'text_encoders\'),), \'type\': (CORE_NODES[\'CLIPLoader\'].INPUT_TYPES()[\'required\'][\'type\'][0],), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'CLIP\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load the selected Gemma encoder without passing a conflicting device keyword.\n    # 不將自訂 device 參數直接傳入原版 CLIPLoader。\n    def load(self, clip_name, type, device):\n        # Leave the core optional device parameter at its default.\n        # 使用原版 Loader 的預設 optional device 參數。\n        return _load_component(\'CLIPLoader\', device, clip_name=clip_name, type=type)\n\n# Use ComfyUI\'s VAE loader for both video and audio weights.\n# Video 與 Audio VAE 都沿用 ComfyUI 的原版模型載入邏輯。\nclass LTX25VAELoaderGPU:\n    # Use the native VAE choice list, including any supported approximations.\n    # 保留 ComfyUI 原本完整的 VAE 清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Add explicit GPU selection to the existing VAE filename input.\n        # 在 VAE 檔名欄位加入指定 GPU 選項。\n        return {\'required\': {\'vae_name\': CORE_NODES[\'VAELoader\'].INPUT_TYPES()[\'required\'][\'vae_name\'], \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'VAE\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load VAE weights with the core loader in the selected CUDA context.\n    # 在指定 CUDA 裝置中呼叫原版 VAE Loader。\n    def load(self, vae_name, device):\n        # Reuse the core VAE loader for the chosen model.\n        # 透過原版 Loader 讀取選定的 VAE 權重。\n        return _load_component(\'VAELoader\', device, vae_name=vae_name)\n\n# Adapt the new-style LatentUpscaleModelLoader without inheriting ComfyNode.\n# 不繼承新版 ComfyNode，避免 LatentUpscaleModelLoader 缺少 INPUT_TYPES。\nclass LTX25UpscalerLoaderGPU:\n    # Supply a legacy-compatible input schema based on public folder paths.\n    # 使用目前 ComfyUI 支援的模型資料夾建立舊式相容 Schema。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Allow selection of the official LTX-2.5 spatial upscaler.\n        # 提供原生 LTX-2.5 Spatial Upscaler 模型選項。\n        return {\'required\': {\'model_name\': (folder_paths.get_filename_list(\'latent_upscale_models\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'LATENT_UPSCALE_MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Call either the new execute method or old FUNCTION method as available.\n    # 依安裝的 ComfyUI 版本呼叫 execute 或原本的 FUNCTION。\n    def load(self, model_name, device):\n        # Keep the original spatial upscaler model-loading semantics.\n        # 保留原始 LatentUpscaleModelLoader 的模型載入邏輯。\n        return _load_component(\'LatentUpscaleModelLoader\', device, model_name=model_name)\n\n# Register the four standalone model loaders.\n# 註冊四種不依賴 MultiGPU 擴充套件的模型 Loader。\nNODE_CLASS_MAPPINGS = {\n    \'LTX25GGUFLoaderGPU\': LTX25GGUFLoaderGPU,\n    \'LTX25CLIPLoaderGPU\': LTX25CLIPLoaderGPU,\n    \'LTX25VAELoaderGPU\': LTX25VAELoaderGPU,\n    \'LTX25UpscalerLoaderGPU\': LTX25UpscalerLoaderGPU,\n}\n\n# Expose readable names in the local ComfyUI server.\n# 在 ComfyUI 本機服務顯示易讀的節點名稱。\nNODE_DISPLAY_NAME_MAPPINGS = {name: name for name in NODE_CLASS_MAPPINGS}\n'
(DUAL_NODE_DIR / '__init__.py').write_text(DUAL_NODE_SOURCE, encoding='utf-8')
print('Standalone GPU loader + targeted upscaler fallback installed:', DUAL_NODE_DIR)
print('✓ 第 4 節完成：模型與雙 GPU Workflow 元件已準備完成。')


5. 啟動 ComfyUI


In [ ]:
'''
Check and start the persistent ComfyUI service.
檢查並啟動可重複使用的 ComfyUI 服務。
'''
import requests
import time

'''
Check that all required dual-GPU loaders are registered.
確認雙 GPU 模型載入節點都已註冊。
'''
def loaders_ready():
    '''
    Use the ComfyUI node registry as the health check.
    以 ComfyUI 節點清單確認服務可用。
    '''
    try:
        for name in ('LTX25GGUFLoaderGPU', 'LTX25CLIPLoaderGPU', 'LTX25VAELoaderGPU', 'LTX25UpscalerLoaderGPU'):
            response = requests.get(f'http://127.0.0.1:{COMFY_PORT}/object_info/{name}', timeout=3)
            if not response.ok or name not in response.json():
                return False
        return True

    except requests.RequestException:
        return False

assert NODE_CACHE_MODE in ('classic', 'none')
STARTUP_FLAGS = [
    '--reserve-vram', str(GPU_VRAM_RESERVE_GIB),
    '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
    '--fast-disk',
    '--disable-comfy-compiler', '--disable-pinned-memory',
    '--disable-async-offload', '--fp16-unet',
]

'''
Estimate available RAM using host and cgroup limits when present.
同時考量主機與容器記憶體限制，估算實際可用 RAM。
'''
def ltx_available_ram_gib():
    memory = {}
    for line in Path('/proc/meminfo').read_text().splitlines():
        if ':' in line:
            key, value = line.split(':', 1)
            memory[key] = int(value.strip().split()[0])
    available = memory['MemAvailable'] * 1024
    cgroup = Path('/sys/fs/cgroup')
    limit_file = cgroup / 'memory.max'
    current_file = cgroup / 'memory.current'
    if limit_file.is_file() and current_file.is_file():
        limit = limit_file.read_text().strip()
        if limit.isdecimal():
            used = int(current_file.read_text().strip())
            cache_file = cgroup / 'memory.stat'
            inactive_file = 0
            if cache_file.is_file():
                for line in cache_file.read_text().splitlines():
                    if line.startswith('inactive_file '):
                        inactive_file = int(line.split()[1])
                        break
            available = min(available, max(0, int(limit) - used + inactive_file))
    return available / 1024**3

CONFIG_ID = ('ltx25-t4x2-lowram-v3', NODE_CACHE_MODE, GPU_VRAM_RESERVE_GIB)
previous = globals().get('COMFY_PROC')

'''
Reuse a healthy service started with the same configuration.
相同設定且正常運作時，直接沿用目前服務與模型。
'''
reuse_server = (
    previous is not None and previous.poll() is None
    and globals().get('COMFY_CONFIG_ID') == CONFIG_ID
    and not globals().get('LTX_CODE_UPDATED', False)
    and loaders_ready()
)
if reuse_server:
    print('ComfyUI 已在執行，保留模型快取。')
else:
    if previous is not None and previous.poll() is None:
        '''
        Request a graceful termination.
        要求舊服務正常結束。
        '''
        previous.terminate()
        previous.wait(timeout=20)

    '''
    Avoid starting a second server on an occupied port.
    避免在已被其他 ComfyUI 占用的連接埠啟動第二個服務。
    '''
    if loaders_ready():
        raise RuntimeError(f'{COMFY_PORT} 已有 ComfyUI 服務；請停止舊行程，再執行此 Cell。')
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(COMFY_PORT), *STARTUP_FLAGS]
    server_env = os.environ.copy()
    server_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)
    server_env['PYTORCH_ALLOC_CONF'] = 'backend:native'

    '''
    Capture server logs for subsequent crash diagnosis.
    儲存 ComfyUI 日誌，方便後續分析中斷原因。
    '''
    with COMFY_LOG.open('w', encoding='utf-8') as stream:
        COMFY_PROC = subprocess.Popen(command, cwd=COMFY_DIR, env=server_env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(180):
        '''
        Detect startup failures before timing out.
        在逾時前先檢查服務是否啟動失敗。
        '''
        if COMFY_PROC.poll() is not None:
            raise RuntimeError(COMFY_LOG.read_text(errors='replace')[-6000:])
        if loaders_ready():
            break
        time.sleep(2)

    else:
        '''
        Provide recent logs instead of silently failing.
        顯示最近日誌以便排查。
        '''
        raise TimeoutError(COMFY_LOG.read_text(errors='replace')[-6000:])
    COMFY_CONFIG_ID = CONFIG_ID
    print(f'ComfyUI 已啟動：Dynamic VRAM / cache-{NODE_CACHE_MODE} / reserve-vram {GPU_VRAM_RESERVE_GIB} GiB。')

print('雙 GPU 模型載入節點：正常')
print('✓ 第 5 節完成：ComfyUI 服務及雙 GPU 節點已就緒。')


6. 圖片上傳工具與路徑管理


In [ ]:
'''
Save local images for I2V and FLF2V while preserving their filenames.
儲存 I2V 與 FLF2V 的本機圖片，並保留原始檔名。
'''
def save_selected_image(change, folder):
    '''
    Ignore events without an uploaded file.
    沒有選取檔案時，維持原本的圖片設定。
    '''
    uploaded = change['new']
    if not uploaded:
        return None
    if isinstance(uploaded, dict):
        filename, item = next(iter(uploaded.items()))
    else:
        item = uploaded[0]
        filename = item['name']
    filename = Path(filename.replace('\\', '/')).name
    destination = ROOT / 'ltx25_images' / folder / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(bytes(item['content']))
    print('已選取：', destination)
    return str(destination)

print('✓ 第 6 節完成：圖片上傳工具已就緒。')


7. T2V：文字生成影片


In [ ]:
'''
Choose duration and resolution for this video task.
設定這個 Task 的秒數與輸出尺寸；解析度或長度增加會提高 GPU／RAM 用量。
'''
SECONDS = 10
WIDTH, HEIGHT = 288, 512
SEED = 42
IMAGE_STRENGTH = 0.7  # 0.0～1.0

'''
Define the prompt specifically for text-to-video generation.
設定只供文字生成影片使用的提示詞。
'''
T2V_PROMPT = '''
電影感多視角鏡頭序列，總時長 10 秒。主角始終是同一位烏黑直長髮、氣質溫柔的韓國年輕女大學生，身穿米白色寬鬆針織開襟衫，肩背帆布托特包；人物臉部特徵、服裝與髮型保持一致。
場景是清晨的櫻花林蔭大道，兩側盛開粉紅櫻花，淡淡晨霧籠罩，柔和金色陽光穿過樹梢，花瓣隨微風緩緩飄落。
0至3秒（側身遠景）：她靜靜站在櫻花大道旁，微微側身眺望遠方，鏡頭平穩取景。
3至6秒（正面中景）：鏡頭流暢移動到她的正前方並緩緩推近，她轉向鏡頭，神情自然柔和。
6至8秒（臉部特寫）：鏡頭繼續推近，她輕輕抬眼直視鏡頭，露出甜美而溫暖的微笑。
8至10秒（微笑與眨眼）：維持正面特寫與眼神接觸，她在最後一秒俏皮地單眼眨眼（wink），隨即保持微笑直到最後一格；微風吹動髮絲。
35mm 底片美學、連貫自然運鏡、浪漫淺景深、柔和粉彩色調、自然皮膚紋理與流暢動作。
'''.strip()
T2V_RUN_NAME = 'clip01'
ACTIVE_JOB = {'mode': 't2v', 'prompt': T2V_PROMPT, 'run_name': T2V_RUN_NAME}

'''
Confirm which mode the generation cell will use.
確認生成 Cell 將使用的模式。
'''
print('目前模式：T2V | 輸出名稱：', T2V_RUN_NAME)
print('✓ T2V 模式設定完成：尚未生成影片。')


8. I2V：上傳一張圖片


In [ ]:
'''
Choose duration and resolution for this video task.
設定這個 Task 的秒數與輸出尺寸；解析度或長度增加會提高 GPU／RAM 用量。
'''
SECONDS = 10
WIDTH, HEIGHT = 288, 512
SEED = 42
IMAGE_STRENGTH = 0.7  # 0.0～1.0

'''
Define the prompt specifically for image-to-video generation.
設定只供單張圖片生成影片使用的提示詞。
'''
I2V_PROMPT = '''
The person in the photograph begins to walk naturally, smooth cinematic camera movement.
'''.strip()
I2V_IMAGE_PATH = globals().get('I2V_IMAGE_PATH')
ACTIVE_JOB = {'mode': 'i2v', 'prompt': I2V_PROMPT, 'first_path': I2V_IMAGE_PATH}

'''
Keep this mode's selection separate from later mode changes.
將本模式的上傳狀態與後續模式切換隔離。
'''
i2v_job = ACTIVE_JOB

'''
Update this I2V job when a local photo is selected.
選取本機照片後，更新此 I2V 工作設定。
'''
def on_i2v_upload(change):
    '''
    Update the reusable I2V image path.
    更新下次也可沿用的 I2V 圖片路徑。
    '''
    global I2V_IMAGE_PATH
    selected = save_selected_image(change, 'i2v')
    if selected:
        I2V_IMAGE_PATH = selected
        i2v_job['first_path'] = selected
        print('影片名稱：', Path(selected).stem + '.mp4')

i2v_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇 I2V 照片')
i2v_upload.observe(on_i2v_upload, names='value')

'''
Display the control for selecting a local image.
顯示本機圖片選取按鈕。
'''
display(i2v_upload)
print('目前模式：I2V | 圖片：', I2V_IMAGE_PATH or '請上傳照片')
print('✓ I2V 模式設定完成：請確認圖片已上傳，再執行 Generate。')


9. FLF2V：上傳開頭與結尾圖片


In [ ]:
'''
Choose duration and resolution for this video task.
設定這個 Task 的秒數與輸出尺寸；解析度或長度增加會提高 GPU／RAM 用量。
'''
SECONDS = 10
WIDTH, HEIGHT = 288, 512
SEED = 42
IMAGE_STRENGTH = 0.7  # 0.0～1.0

'''
Define the prompt specifically for first/last-frame generation.
設定只供首尾影格影片生成使用的提示詞。
'''
FLF2V_PROMPT = '''
Smooth cinematic motion transitioning naturally between the opening image and ending image.
'''.strip()
FLF2V_FIRST_IMAGE_PATH = globals().get('FLF2V_FIRST_IMAGE_PATH')
FLF2V_LAST_IMAGE_PATH = globals().get('FLF2V_LAST_IMAGE_PATH')
ACTIVE_JOB = {
    'mode': 'flf2v',
    'prompt': FLF2V_PROMPT,
    'first_path': FLF2V_FIRST_IMAGE_PATH,
    'last_path': FLF2V_LAST_IMAGE_PATH,
}

'''
Keep the selected FLF2V job separate from the other modes.
讓 FLF2V 的檔案選取狀態與其他模式互不影響。
'''
flf2v_job = ACTIVE_JOB

'''
Save the uploaded image under its correct endpoint role.
依開頭或結尾用途儲存選取的圖片。
'''
def on_flf2v_upload(change, role):
    '''
    Update the reusable first and last image paths.
    更新可供下次使用的首尾圖片路徑。
    '''
    global FLF2V_FIRST_IMAGE_PATH, FLF2V_LAST_IMAGE_PATH
    selected = save_selected_image(change, 'flf_first' if role == 'first' else 'flf_last')
    if not selected:
        return
    if role == 'first':
        FLF2V_FIRST_IMAGE_PATH = selected
        flf2v_job['first_path'] = selected
    else:
        FLF2V_LAST_IMAGE_PATH = selected
        flf2v_job['last_path'] = selected
    print('開頭：', Path(flf2v_job['first_path']).name if flf2v_job['first_path'] else '未選取')
    print('結尾：', Path(flf2v_job['last_path']).name if flf2v_job['last_path'] else '未選取')

flf_first_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇開頭照片')
flf_last_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇結尾照片')
flf_first_upload.observe(lambda change: on_flf2v_upload(change, 'first'), names='value')
flf_last_upload.observe(lambda change: on_flf2v_upload(change, 'last'), names='value')

'''
Display both controls in this FLF2V mode cell.
在 FLF2V 模式 Cell 中同時顯示兩個上傳按鈕。
'''
display(flf_first_upload, flf_last_upload)
print('目前模式：FLF2V')
print('✓ FLF2V 模式設定完成：請確認首尾圖片已上傳，再執行 Generate。')


10. 生成影片


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    spec = importlib.util.spec_from_file_location('ltx25_kaggle_engine', ENGINE_PATH)
    engine = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(engine)
    engine.KEEP_MODELS_RESIDENT = True
    if 'ACTIVE_JOB' not in globals():
        raise ValueError('請先執行 T2V、I2V 或 FLF2V 的模式 Cell。')
    job = ACTIVE_JOB.copy()

    '''
    Read the selected mode and its independent prompt.
    讀取目前模式及其專屬 Prompt。
    '''
    MODE = job['mode']
    PROMPT = job['prompt']
    image_name = engine.ensure_placeholder_image()
    last_image_name = None
    if MODE in ('i2v', 'flf2v'):
        first_path = Path(job['first_path']) if job.get('first_path') else None
        if first_path is None or not first_path.is_file():
            raise ValueError(f'{MODE.upper()} 請先在模式 Cell 選取開頭照片。')
        RUN_NAME = job.get('run_name') or first_path.stem

        '''
        Upload the first frame to the ComfyUI input folder.
        將開頭照片放入 ComfyUI 的輸入資料夾。
        '''
        image_name, _ = engine.save_upload_bytes(first_path.read_bytes(), first_path.suffix, RUN_NAME)
    else:
        RUN_NAME = job['run_name']
    if MODE == 'flf2v':
        last_path = Path(job['last_path']) if job.get('last_path') else None
        if last_path is None or not last_path.is_file():
            raise ValueError('FLF2V 請在模式 Cell 選取結尾照片。')
        RUN_NAME = job.get('run_name') or f'{first_path.stem}_to_{last_path.stem}'

        '''
        Upload the final frame independently of the first frame.
        獨立上傳結尾照片，避免覆蓋開頭照片。
        '''
        last_image_name, _ = engine.save_upload_bytes(last_path.read_bytes(), last_path.suffix, f'{RUN_NAME}_last')
    import secrets

    '''
    Resolve one seed per run without overwriting the notebook setting.
    每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
    '''
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    OUTPUT_MP4 = ROOT / f'{RUN_NAME}.mp4'
    LAST_FRAME_PATH = ROOT / f'{RUN_NAME}_last_frame.png'
    print('MODE:', MODE, '| MP4:', OUTPUT_MP4.name)

    '''
    Preserve the audio batch-size setting required by the original workflow.
    保留原始 Workflow 所需的音訊 batch size 設定。
    '''
    queue_original = engine.queue_prompt

    '''
    Normalize audio batch size before submitting the prompt.
    送出 Prompt 前統一音訊的 batch size。
    '''
    def queue_with_audio_batch(workflow):
        '''
        Inspect the workflow for audio-latent nodes.
        尋找 Workflow 中的 audio latent 節點。
        '''
        for node in workflow.values():
            if node.get('class_type') == 'LTXVEmptyLatentAudio':
                node['inputs']['batch_size'] = 1
        return queue_original(workflow)
    engine.queue_prompt = queue_with_audio_batch
    params = {
        'mode': MODE,
        'prompt': PROMPT,
        'width': WIDTH,
        'height': HEIGHT,
        'seconds': SECONDS,
        'fps': FPS,
        'seed': RUN_SEED,
        'image_name': image_name,
        'last_image_name': last_image_name,
        'audio_name': engine.ensure_placeholder_audio(),
        'image_strength': IMAGE_STRENGTH,
        'vae_tile_size': VAE_TILE_SIZE,
        'vae_tile_overlap': VAE_TILE_OVERLAP,
        'vae_temporal_size': VAE_TEMPORAL_SIZE,
        'vae_temporal_overlap': VAE_TEMPORAL_OVERLAP,
    }
    available_gib = ltx_available_ram_gib()
    print(f'可用 RAM：約 {available_gib:.1f} GiB', flush=True)

    '''
    Reject high-risk jobs instead of allowing the OS to kill ComfyUI.
    可用 RAM 明顯不足時拒絕生成，避免讓作業系統直接終止 ComfyUI。
    '''
    if available_gib < LTX_MIN_AVAILABLE_RAM_GIB:
        raise RuntimeError(
            f'主機 RAM 不足：可用約 {available_gib:.1f} GiB，建議至少 {LTX_MIN_AVAILABLE_RAM_GIB:.0f} GiB。'
            '請先重新啟動 Kaggle Kernel、避免同時執行其他模型，或縮短影片秒數。'
        )
    start_time = time.perf_counter()
    try:
        generated_path = Path(engine.run_once(RUN_NAME, params, MODEL_TIER))
    except Exception:
        print('ComfyUI exit code:', COMFY_PROC.poll() if 'COMFY_PROC' in globals() else 'unknown')
        events_path = Path('/sys/fs/cgroup/memory.events')

        '''
        Show OOM kill evidence without assuming the process died from OOM.
        顯示 OOM Kill 訊號，不直接假定中斷原因。
        '''
        if events_path.is_file():
            print('Cgroup memory events:\n' + events_path.read_text())
        print(COMFY_LOG.read_text(errors='replace')[-4000:])
        raise
    shutil.copy2(generated_path, OUTPUT_MP4)
    elapsed_seconds = time.perf_counter() - start_time
    print('MP4:', OUTPUT_MP4)
    print(f'生成耗時：{elapsed_seconds / 60:.2f} 分鐘（{elapsed_seconds:.1f} 秒）')

assert OUTPUT_MP4.is_file() and OUTPUT_MP4.stat().st_size > 0, '影片輸出不存在或為空。'
print('✓ 第 10 節完成：影片已生成並儲存 MP4 檔案。')


11. 預覽與下載


In [ ]:
'''
Confirm that the generated MP4 exists.
確認生成的 MP4 檔案存在。
'''
assert OUTPUT_MP4.is_file(), '請先執行生成影片 Cell。'
display(Video(str(OUTPUT_MP4), embed=True, width=448))
display(FileLink(str(OUTPUT_MP4)))
print('✓ 第 11 節完成：預覽及下載連結已顯示。')


12. 擷取最後影格（選用）


In [ ]:
'''
Verify output and FFmpeg tools before extracting the final I2V frame.
擷取 I2V 接續影格前，確認影片已完成且具備 FFmpeg 工具。
'''
assert OUTPUT_MP4.is_file(), '請先成功生成 MP4。'
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg／ffprobe；請先安裝。'
import json

'''
Read the frame count from the saved MP4.
從已儲存的 MP4 讀取影格總數。
'''
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(OUTPUT_MP4)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUTPUT_MP4),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)

'''
Preview and link the extracted image.
顯示擷取圖片並提供下載連結。
'''
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：MODE = 'i2v'，IMAGE_PATH = '{LAST_FRAME_PATH}'")
print('✓ 第 12 節完成：最後影格已擷取並提供下載。')


13. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'width', 'height', 'seconds', 'seed', 'image_strength']
REMOTE_IS_LTX = True

'''
Provide safe defaults if the shared video-parameter Cell was not run in this Kernel.
即使本次 Kernel 尚未執行共用影片參數 Cell，也能提供相同的預設值。
'''
LTX_REMOTE_DEFAULTS = {
    'SECONDS': 10, 'FPS': 24, 'WIDTH': 288, 'HEIGHT': 512,
    'SEED': 42, 'IMAGE_STRENGTH': 0.7, 'MODEL_TIER': 'q4km',
    'VAE_TILE_SIZE': 256, 'VAE_TILE_OVERLAP': 32,
    'VAE_TEMPORAL_SIZE': 16, 'VAE_TEMPORAL_OVERLAP': 4,
}
required_setup = ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG',
                  'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB')

missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError('請先執行 Notebook 的環境、模型與 ComfyUI 啟動 Cells；缺少：' + ', '.join(missing_setup))

if not Path(ENGINE_PATH).is_file():
    raise RuntimeError(f'找不到 LTX 引擎：{ENGINE_PATH}；請重新執行模型與 Workflow Cell。')

REMOTE_SCRIPT = zlib.decompress(b'x\xda\xb5Z[W\x14Y\x96~\xe7W\x9c\xb7Lz$\xb5h\xab{\x15S>P\x9aRLq[\x80v\xcf8\xaeXA\xe6I2\x8a\xc8\x88\x9c\xb8\x80\x8c\xedZ\x89K\x04\x04\x84*\x01\x0b\xa1\x0b\xb5E\xb3T.^P\x04\xc4\x1fS\x19\x11\x99O\xf5\x17f\xefsN\\2H\xbb\xfae|\x108y\xe2\xc4\xbe~\xfb\xdb\xfbd"\x91h\xea\xd2\xe5,\xb1\xf2\x94\xe8\x862\xach\xb2J\xac1\xbd\xc5\xb4\xe4aJ\xba\x06\xff\xda\xd2\x9a\xfa\x92\x8c\xe9\xc6HN\xd5\xc7\x08\xd5`\x0bM5U\x8ff\x9c\xc9M\xe7\xee\x86\xbb\xfe\xd2{p+\xd8X[[\xad=\xb8\xebn\xef\x91\xbf\xf8\x8f8G\xcb\xee\xbd\xbb\xbf\x96n6%\xe0mf\x91f\xc89\xa2\x14\x8a\xbaa\xa9\xcaP\xca\xb6\x145\x85\xabR\xce\xd0\x0bRNQ\xa9\xa4\xea\x19\xd9Rt-\x99P\xadk\xad_J#\xf2\xf00,\xf3\x97\'N\x91tOGgOZ\xeak\x1f\xfc\xb6\xb9\x89\xaf\x9e<\xb3\xa0gmx\x88\x9d\x8a\xe7\'\xf1\xbff&@J\x05\x9d\xa9\x91\xa2\xd7\xe0\xb5|_\x92\x1f\xd3\xdc\xc4\xa4\xec\xa7\x96\xach\x04>\xa2\xaaI\xe4\x8c\xa1\x9b&\xc9\xe8\x85\xa2J-\x9a%\xc3T\xa3\x06\x93\xd0\xfcwb\xc2\xcbT"\xab\xa8k\x81\x16tc\xbc\xa5hP\xd3\xb4\r0i.\x87\xafR\xb4\xe1TS\xe5\xd3\xdf\xbd\xe5\xd5\xea\xfb\xb2\xfb\xf2\x91\xb7\xb4\xe1N/\x82\xe1\xdc\xf2#\xe7\xe7Y\xe7\xd3\x0bga\xe5\xb7\xa3\xb9\xca\xfeSgr\xa2Z~U-\xff\xe4\xde~W{\xb1\xe4\xfc\xe3\x9esg\xad\xb2?\x0b\x9b\xabS\xcf\x9d\xd9e\xd2\xcb\x0f\xf5-*\\\xf2]:\xdd\'u\xf7^Hw\rH\xfd\xe9\x81\xce\x0b\xe9\x9eA0\xca\xa0aS_\'\xf0\xb3\xac\x81\x99\n\xb6%\x0f\xa9\x94\x98\x9a\\4\xf3\xba\x05b\xb2\x00Pe\xd3"h\x13\x1b\x95D\xe5I\x86\xaaj\xaa\t\x84s\x8e\xef\xbb\xeb%\xe7x\xce\xd9x_}4\'D?Z \xe7a\x07\xa9\x96\xb7\x9c\xed\x07\xa0@m\xe2\x9339\xcf\xd5\xab\xeco\xd5JK\xd5\xc7/*\xfbw\x9c\x9bo\xdc{\xf3\x95\x8f\xeb\xee\xda[\xd0\xc1\x97]\xc9\x91D\xfb\xf9\xc1\xce\xcbi\xe9?z\xbfI\x10\r\x84\x01\xbb\x0f\xab\xfa\x90\xac\x9a\xc9\xe6\xb6\xa6&\x02\xffp+\xfe\x1c\xa0*\xcd\x80\xb8\xe0\xee@:2Ds:\x98\x1a\x02\xd6\xb0\xc0\xd0\x11\xe7\xa4\xd8C\xb5\x95Y\xe7\xd9,\x17\xc9\x99\x99\x07!\x9dO\x93\xb5\x87\xd3\xce\xe44\xd7\xc5\x99|\x07\xa2V\xf6KNi6\xa2\x14\xca\x18}\xb9!+&%\x97e\xd5\xa6i\xc3\xd0\x8dd\xa2\xfab68\x84\x0c\xb6^\xfe\xb54\xd1\xd9z\x99\xb8\xd3+\xe4b\xd7E\xf8\xad\xdeJp`B\xc4\xd7y\xbd8\xceL.g,e\x94\x92\xef\xf5!b\xe9\x04\xe2f\x94j\x168\xc2\xa2\x06\x19S\xb2\xc3\x14\x1c\x82K&\xc18&\x99\xbc\x8c\xde\x1e\x86g\x15\x93\x186\xa8X}2U\xfd\xc7Gom\x1bt\x83\xf79\xef7\xc1\xca1\x7f\x80\xdb\xbc\x839\xee\x86\xca\xc1l\xe5\xf0\x9d\xbb\xf4\x01\xdc\x00\xc9\x1b\x04\xa3\xef\x12\x14\xe5\x1c\t\x9d\x92\xca\x80\xac\xc9\xe6H\x0c\xa1\xdc&\xf3\x84\x1f%\xb2\x96%\n\xc8\xa8hYZ\xa4\xf0\x1f\xe8P\x04y\x8b\x16\xc8\xb7]\x82\xf0\xe1\xf2qc8\x0bw\xc0\xe4\xce\xee\xb4\xf3\xea%\xe9c\xdb\xfc\x97c\xfc\xc2\xdbA\x86+\t<9q\xb5\xa9\xaf\xbf\xb7\xbbo\xd0_\xe4\xa7\xc22\xdb\xfe\x1d\xa5E"\x93\xa2*gh^W!\xa5!\xba\x11\xb7 "\x88E\xafY-\xba\xa6\x8e\xd7E\x84\xf7v\xc9]\x99r\xb6\xeesY*\x1f?yK\xe5\xda\xc3E\xb0\x98\xf7\xcb\x81\xb7\xfa\xd1Y_\xf1f\xa6\x82\x00\xc5\xe3$M. \xc0\x88L\xa3\x1a\xe6\xb6\x14y\xab\xc4\xb6\x81\x910\x83\xa4\xbagz X}\xdb\xfd\x8f\xad@\xa02\xa8\x05+\xa1\x1bCi\xa9\x02\xebB\xfc\x96\x8c\xaee\x15\x94W\x18\x183\xf0y\xf5\xe9\x04\x97\x96K\xc8\xbd\xc6\xb5\xc0\x98\xde}\x15\x865KW\x88\xfb\xda\xc3-o\xf2YT\x9d\x1ca&\x86\x1cK&\x94\xd6Q\xc0\xd2DN\xcd\xc1/\xf1T\xeb\x80\xc8\xabstN1\x00\x1d\xb8\xc0E\xd9\xca\xf3\xe4\x12\xaf\x9a\xd8G\x1f\x03B\xed\xbd\x87\\\xe3\x02V\xdf\xef8\xc7?\xc4\xb3\x88\x1d#\xe1\x01`\x9c>\xf8\x91d~\r\x97\x13W\x9b\t\x88\t\xab)\x88\xfed\xf4\x93f\x02hL\x85I\xa3\x87\xf6\xd3\xef\x11\x17\n\x8ai\x06V5O\x00CG\xdf\xa5\x13\xe0\xe0\xac\x97\xd9z\x0c$\xdc\xd9\x1f\xbd\xbde\xef\xe8\xc0\xd9\xfdA\x98\x1b\x94cI\x15W\x08d\x8d\xe8\x04Y\x89\xe2A-eX\x16~\x92RLV\xdb\x02L\x8b\x9e\x81\xff:5\xd32\xec\x0c\xb7\xbamB,\x00 \xd8E5\xa8\xce"d\x8a\x00\xd8z*x\xcc]X\xf4\x9e\x1c\xf0\xc0\xa8\x96&\xc1\xfb<\xcd\xb9\'B\xef7z\xe5\tD\xcb%\xaecx\xa4\xecb\x91\x1a\xc9\xe6\x1bD \xdcz9\n\xf6\xdc\xdb\xb1\xf0Bd\x8b\x9e\x7f\x81\x1a\x08l(9b\xa2n[E\xdb",)\x18\x92\xb1\xd8\xd2m#C\xb9\xbbN\x11[S\xa1jB:\x1bPD-\x8e\x89l?\xd8\x14\xd2~T\xc9\xd2,\xd7\x9b\x81\xec\xcc,\x00\x98(4L\xfb\xca\xf1m\xf7`\xd1}\xbe\xe4,\xce3\xe4{\xe8\xbd\xd8\x89n\xf0\xe6\xcb\x1e\xe8\xc3\x00\x12\xf6x\xe5Wq_\xf6_\xea\x91z\xda\xbb\x05\x04\xf1\xf8\x03\x90e\xb9\x0c\xd1\x07N\x8d8\xd4\xb4h\xa1^\xe7K\xa1\xb7x\xb6\xe4\x0c\x94\x1f\xfc\x88K\xe7\xf5Bn\xfcR\'\xe4\x1eZ"\xc7\x80C\x04\xe1\xeeT\xd4\x9a\xee\xd21p\xab`?F\xde\xd1>\xacT\xdfL\xb9+\xab\xce\x93\xe3\x13!\x18\x00\xce)"\x858e\xca\xa3T\xe2\x11$\r\x8d[\xd4LF\xa47h\xb0\xda|\xaaN-;\x97S\xae\x9d\nl\x01\x04\x0b\x92.\x86\x0e\x97L\xeeZz\xad\xa8*\x19\xc5\xc2\xeaG2\xaaR\xe4\x1e\x1b\xcbS\r\xe2_\xa0\x05\xf8\xcf\x84\x88R\x15\xdf\x7f\xb8\xd9}\xfd\xa3\xbb>\xc3}\xc63\x0c\xe9\x0f\x0f\xe3\xa9\xe7\xd5\xf2M\xace\x1f_\xc1\xfa\xbf\xe2\xaa+\xa1\x9b\xae\xd6\xa3\xadL\xb2\n04\r\x12\x0b\nS=\xe62\xadO#f\xb7pOE\xd1\xa1\xf6t\xc5\xd9=\x06\x11\xdc\x87G<\xfak\xeb\xa5\xda\xa3y\xe7\xc9\x8a;7\x05@\xe0\xed-\xc2\x86\xc6\xf8z\xee\x9c\x8f\xabmq\xa4\x12\x01R\'L\x00\xb2C\xe3\x02t\xb1\x02\xf8\xd9?d[\x96\x8fW\xa2\x9a\xbe|Y9\x98C\xba27S\x9b^\x0e\xe07.R\xf4\xcd\xac4\x9d\xc0\xdd`5\x0e\xbb\xe1\x07\xbf\x87\xba@)m\xcd\x84r\x01\x16\xf5\x118\xc7\xfa\x08\xa6\x1c\x97\x9b\xbb\x9b\x1b\x0e\xf8\x91;s\\\xd9\x9fw\xa6w\xa3\x12\xbb\xab71?\xe7\xdf\xb8\xb3\xb3\xce\xd4Amn\xa7\xfa\xfcI\x03\xb8\r5\x89\xa1m\xf0\xc1\xef\x82m\x1a\xe2\x16\t\xfeX^\xc9\xe4EF\xc2aB\xfc8\xba\xf2:\xd0\xc8\xb8\xbf\x8b\xa8\tN\x04\x11G\x1b\x81h\xec\xcc8\x88\xf60\xf8\x80\x80(\xc8\x9a\r\x06e\x19f3\x0b\x0f\xe9\xa8?\x0fe\xd0\x14C\xdf<\x85\x86\x18AN\x84\xcf\xc4`\x94+\xc5\xc59\t\x9e\xce\xe4/\xce\xd1^P\xea\xfe\xbfQ4q=\x06\xa47$K\x97\xae\x87\x0edk\x89\x7f\x02\xaf\x18_<i#\xa4\x13\xb8\x9ehe"\xf8\xcb\x15\xe7b\xf3\xda\x18\xb5z\xc0\x91\xabOoW\xef\xcd\xc6I\xd3\x89\x0c\xfa\x97\xb16\xd4\xa5\x1ej#:\n\xa4\x05k\xf8&\xbb!\xe1\xc7~\x97\xd0\xc1\xe1\x08A,\x07\x1de\x9e\xfc\xb1\xb5e\x08\xc0\xd6\xa4\x00\x15\x8c\xd92\xa0m\xf9\x02c\xd7\x00\xc0\xa3`\xb6,#\x8b\x90l\x9c\xae\xe0\xa7\x90X\xee\xcc\xbcsx\x00\x16pWv\xb1_\x17\'\xd5\x1e\x94\xdd_6\xbc\xf2\xb6\xb3\xb5\x18R^l\xa3\x11\x83\x0cj\x99>\x9a\x9a\xba\n\xe5\x1c\x93\x8d\xbd\x1d\xf8\x01\xb6\x1f\xd0\xa3Xy\xa8\xedD\x1f\xa5\xc6\x98\xa1X\xbc7\xa1\x98\x8etH\xd7G`\xb7e\xb1\x94rw\x16\x82^\xc3}u\x00\xa2\xf1\x8e\x8b\xbf\x9c\x81\xffm\xc0\x04p\x83\xb3\xf3\x02\xd2Z<\x8fe\xe0\xee\x06P\x99\x90}1\x81\xc0`\x03\xe9\xf4\x05\xf0\x80\x104e@\x172D\xa1\x0bO\xb6\xfe\xe1\x0f\x7flep\xc6\xb7\x9cC#0\x10S4+\x89k\xcd\xf1"\x81\x05\n\xb4\x06\xcd\x98z\x82:\x0e\xd9\x8a\x9a\xf5UR\xb4\x1c5\xa8\x06\x84\xc5\x9f\x86\x80\xa5\xd7\xcb\xc2\xacw\xcb\xd5\x17+<+\xdc\xbd\t\xaf<+Z\xcd\xc9i\xf7\xf9cwc\x93\xab\xe9\xed\xdcr\xd6\xe7}%d\x13h\x9eE\xce\x90\xaf\xcf\x91@\xa1\xaf\t\x13\x1f\x889\xfb\x93\x93\xfa\xca!8\xee\x08vV\xa7\xde\x90\xb3\xad_\x9d\xfd\xeaO\x7fn\xfd\xeaKx\x01\x80)\xb6.7\x0f@GD\x10\xae\xd8@^\x1f\xf3\xfbK\x04\x0e\xa6\x94\xa9#\xb9\x02+\x01\xfdB\xdfe\x00\xb9\x87\x10\'\x80]e\xed\x0c\xec`}\'\x94\xbeG;\x80z\xce\xce\xd3\xda\x835\x91\xf4P[|?\xb9+\x1f*\xc7\x9f*\x1ff\xa1\xab\xacM\xcd{\x0b\xc7"\x8a\xf8Lco\xd1\xfdy\xdd\xd7\xb0h\xa0\xc5s\x89h\xa3\xc9\x84\xf9\xed\xe8\xc1u_\xe5\x1baW\xecs`A\x17\xfd9\x10\xb0\xc2\x1cv\xc4\x8d;\xcf\x00\xff\x88\x0c>\x1b\xd1\xf41\xec\xf3\x1ec\x88\x05\r\'\x074\x90\x18}r[\xe4\x02\xd2\xaa\xa9\x03\xce2\xaa\xd3S\xbc\xd6#\xdd\n:\x17\x94\xaa\xf7\xd2`\xdf\xa5A\xa9\xbb\xef,\x04[\x7fo\xef 9]\x97\xb2\xa9B\xf1l\xa2\xa9\xab}`P\xba\xd8\x0f\x0bl,\xd5x+\xcbn\x89\xa3RQ\x1b\x8e;+\x93\xd7M\xc8\xe7(W\x16\x81(\xb8\x16t\x9eH\xaa\xb1a\x81v\xc0g*\x10\x84\xd5\xd2}\xac\x9e\x90\xd8\xac\x99\x99\x98\xf5\xb6\xefC\x00rG\xd6J\xabh|N#\xa7\x0e\xb8)\xea=\x94@\xd6\xd2\x06\xdd \xfe\x84\xd0\xfb\x1b\x01u\xf1\xefP\xf9\x14\xca#\x1c\xd5\x07hD\rA\xefe;\xab\xe8dH\xb62\xf9\x16S\xf9_\xea\xa7<C$\xc8\xae\x80\xd5\x04\xf3\xc50\x7f\xf8L\x8c\'x81tgJ\xc0\xb5@\xe0\xda\xc6\x9bj\xf9\x0e?\x9b\xb0\xb3\xebQ\x00\x10\xcf\xa6Rpn\x80\xc6|\x9d\xcf\n\xb8\xc4=\xbaQ\x90U<!".?\xd2o\x13\xed\xa1\x82b\x05\x00\xe6\x8f/j\xa5\t\xb0\x99\x18S\x10\x9c\xb1\xec\xbd\x02\xe0\xe2\x92\xa1\xc1\xc3\x93|\xa9\xb24G\xb8\x04\x88\x8e\x12{\xa1\xc4\xb6%}\xcd\xe3\xdd64\x7fE*z\xbf`\xd6\x8a\x14\x95=\xdc\x82y\xa9AD@\xc4\x9b~\xcb0\x0b\x1c*4Ye\x7f\x0b\x85\xe1\xca\x89\xed\xdeN\xa9v\xf8\xf3\x89\x06\x9c\x91\xa5,\xe2Y\xe8\x88Qd,\xe6g\xe9\x12\x9f\xb3\xc4\r\x87\xf3P\x08A\x9c$\xa2\xa2Lx\xec\xd2t\x11\xc1!\x8d\x82d\x04O7\xf0&K:\xcc>\x16\x99\xde\xda\xbe\xb3\xfd\xe1s\xdc\n\xe0\x1c\xe5\xe6|"\x03\x99dJ\xd6x\x11\x19\x05\xb2\xed\xae\xc1\xbf^N\x83\x8b\xc6\xbb\x98\xf2\xed(l"\xa2N\xfc4\xbf\x8ba\x15m\\\xcb\xe4\r]\x03\x91\xb2BM\x13\xab)`}\xaa\xee\x01\xd1h\xee\x97\xbc\xbd[\xce\xe2\x9c\xbb\xb5\xc9u\x8a\x8a\xdc\xe8E(\xf7\x95\x04\xa3\x99f\xe2\xea\x95\x043\x81\x84&H\\\x85\xa8\xfd"6\xe5d\x91\xd88e\xa2@\x08\x01\xad\x06NPT\xc5\x1a\'r\xf6{\xdb\xb4\n`\x00\x11&\xdbs\x88\xc9\xcc\xaen\xe9Y\xf5\xf9\'w\xf9-GA\x1e\xd8\xb1\xd4;1\xf3\xa4\x96mh\xa4>\xcf\xc2(\xe6\xb9\xd5\x0e\xbd\xddx\x04\n>\'\x12o\x84\xfd\xb9%6\xbd<\x89b\xd2m\xdeg=\xe0\x9bF\xc3\xc9\x06\xf9\r\xf6k\x9cl\x02\xaa N\xea\xeb\x86\x80\'\xd3\xef\xcb\x1b\xc0\x12\xc8\xc6\x81\xc6)\x1d\x01_\xac\xbb\xe9\x88\x9a\x8a\x01\xa8\x0cpn\x82\x14\xd7\xb9\xe5\xd8\xd8\xb2\x8d\x03)_\x113\xcb6\xc2G\x99buL\xc9Bs\xd5F\xfe\xd2ya\xf0[\xb1\x96\xa7\xcap\x1ew~\x9b\xee\xec\xf8\xd6\xdf\xc9\xbbA\x13\x96\x07\xd2\xe7{{.\x0c\x88\xf5\\\x11\xd7.\xf6\r\x04\xfbh\x16\x16\xfc\xb2*VC\xc2\n\x9fE\xd8+\xff4\xc6iaK\x9c\xe5\xf2}\xdc\xaeb\xcb\xe7\xe7\xa0l\x1b\xb0\xdc\xe8\xab\xa1\xa4\xc3\x03L\xd7\xce\xee\xf6\x8e\xb440\xd8\x9f\xee\xe9\x08\x94\x1e\x95\xa9d\xe1\xad\x0f\xcb\x876r\xb9=-\rvv\xc1\xbe\xce\xffJ\xc7\xf7 \xcfT\xe5bt[\xef\xe5t\x7fW{_t\'E\n+\xabu\'\xa6\xbb\xfbz\xfb\xdb\xbbN\x9c\xea\xef\x8d\x9d\xeco\x0fN\xbf!8J\x9efF\x08\x14j\x8bq\x8f\xcc\xb0\xa1\xdbE\x02u\xff\xc4\xb8\x11\xd8UX\xb29.\x86\x11\xc6\xe7\x8dP\xb6\xab\xaf\x0fq\xaes\xb0\x18\xcc;"7g\x8cBr\xfeX\xd9?\x04\x8e\x85\x80\xb9\xfd\xc1Y-sr\xe6,@kV\xc6\xb7\x07\x94rTVT\xbc\xdc\x91\x86\x15\x9c\xe3\xab\xd65)\\\x83H\xc5\xf5ds@\xcc\xc2\x13\x80\x91yoo\x91\xebu\'\xb4\xa5\xbe\xc8\xdd \x1d\xca7@\x05r\xaam\xe6\xcf\xe1}R\xc0\x9eY\xff\x9f\x87\x98m1\x14s\x04\xbb=\xbc\x03\x80&\x04\xda4h\xc4\xd8\x9d\x98_P{\x070\xe1F\xf0\xaeLL\xb6\xb0Q\xf1\xdfN\xdc\x9f\xee\x02[\xc1\x06\xe0\xdd\x1b\xecW\xf8\x00\x96%\x7f\xd8\xa2m\xdfCz\xbd\xb9\xe5\xbd9\x842\xec\xad\xbdu\xefnz{7\xdd\xad\xc7\xfe\x99\x91AL\xbd)\xbe\xc6+J\xa9\xbb\xb3Gj\xbf\xdc\xde\xd9\xd5\xfe\rD\x0e\xbcW\xea\xe8\xfc\xa6-r\xbd\xd3ok\x96R\x10\xcd{\x80\xe1\xb9\x047?\x93\x94\xcb\x08\xe6\xe2\xd2\xff\x13\xa3!\xd7<<\xa8n\xed\x02\x9a9\xbb?\x90\xeb\x9f\x17!u\x86?\xc2(|PF\xf8\xf8\x15H6tm\xce\xf2\x06^\xfe}\xc7\xaeB\xa1"\x1b\x1aU\x7f-M\x88\x0b\x1e(F\xab7\xc3\xeb\xac\xc3\x15~\xb1\xc8\x1b\x04o\x7f\xdb\xdb\xd8\xe2\xd1\xe5=\xfb\xd1]\xde\x0f^#\\\x19)9\xe1\x0c\x8c\x05x\x81\xca\xec&\x93\xf1Sq\x01\n\xbd\x94\\4\x01J\xd1T\xe0E~#(\xee\xfc\x9e\xba\xfb/kS\x0b\xde\xfeGNJ\x83\x8b_\xcc\x0b\t\x9f\x80\xb0d\x0f\x02\x83\xcdI\x19\x1d,\x8e\xc3f\x11S\xd0\x97\xd4\xbd*,| \x8c\x89LY\xc6\x1f\xd06Q\xf4\x11\xb4\x08\xc3\xc8\x0c@\xf8\x1c\xb8\xc0\xf4\xa5\xc1\xc2\xb7\xfc6H%\xf4\xc4\x93W\xee2\n\xc4I\xb1\xf7\xfa\xd8\xdb}\xc4gL\xde\xdbRm\xee\x96/\xa8e\x8c\x9f\xb8\x0b\x11}x]\x1da\xc3\x18F|\xf0\xeaK\x93\xd9E^xu\x1e\xe6z\xc8\x1c\xa0\x8a\x80\x99\x1a\xdc\x91\x8b\x8b\x07\xe6\xa1x\r\x16\x0e\xa1\xd9\xba\xe9\x9d\xc0`\x9c\xab\xe8\xc0S\x92~\x8fq\x8a\xf0\x82\xc4\xa9|\x17@d\xba\xbf\xb9\xb9\x89^\xcbP\xa8\x95i\xf6\x03|{b,\xc9z~\xae\x1f\x10{\x03\xa0\x0bL\x9aA\x9e\xc8f\x0c\n\x9b\x8fi\xc0-um\x98\xf7\xfeZ0)\xf3\'\xd5\xce\xfb\xd7\xc0C\xdd\x95\xf7\x81\x951.\xca\xb3\xd8\x13\xfe}\xc3{|\x14WM4\x1f\xfe\xf3\xc1;\xb1\xf38\xdf\xdb}\xf1?%(\x99\xe7SE]U\x93\xac\x9fO\x84\xab\x89\xba\x8bb\xde\xde\'l\x8d\xb5\x7f\xf19\x1a\x07\xed.E\xb3\xaf\xf9W\xf4PR-\x03\xce\x05]D\x08\x9a\\S\xb0\xc18k$\x83\x9c\x16\x03$\x869\xee\xd2\x8e\xbb\xf0\x16\xf4\xe3\x83Xqfp[\xef\xde}\xe6L\xbf\x03|\xae\x96\xa7!\xcd\xe0\x97\xb8\xca\xfc6\xb7\xce\x95\x89\xd3\xe6\xb8y:g\x9e\xe6\xc5\xe44\x970\xc5w\xc6Ua\xcdboo7GR\x8au\x85\x8d$\xc4\x1c\x06\xd8\x8e]\x88\xf40\x19\xbcR\xc9*x\x7f\x87\xb7.\xf0\xa0\xb8\x14g\xeea\x07}\xa7\xb0\x0b\xfc;\xd5\xd5\r\xf6\r\x84y\x8e\xab\xce\xc4\x14\x9b\xd1\xa0?\x91%\xae=l0\x8a\x8dh\xf3\xbb3\xd7>t\xb6Hnap(\x08|:?l+f^\xb8\x86i&\xee\xbbuv)\x8a\x99\r\x18d\x86d\\D\x97od\x94\xfbp\xd3\x99/9\xd3\xb7\x03_\xf0\xc2\x80\x15\x93\x01b\xfd89.\x9c\x1f\x89\xbc\x9c\x0bA\xb8vm\xff\xad%\xc8\xbf\xd5\xa9\xca&zx\xbf\x9cl\x8e\xb9\'T\xd2\x00\x9c\x18\x955v\xed\xe0O%\xfd@\x179\x06\xd8\x15\xf5\x86\xbb^\xaa~\xfa\x019\xa6\xbf\xcf\xbd\xbfY}\x0e\x80\xfe,\xd2\x16\xd5\xe7\x0e\xcf\x87\xae\xde\x8e\x88L\x0c\x18\xcds\t\x832j\x96h\xbe\xd2r\xf6\xcc\x993mWO\xc8\x1a\xe9\xe2\x03\x16\xcca\x15\xd1\xd62\xe0\xe9!93" ,\xd2\xabsc\x82m\xc3M\r\xbf.\x11\xfb\xb6\x03\xbc\xd0V\x199\xc2i\n\xc8\xcf\x19\xb8\xc9\xbf\x8e"\xe6\x1e\xfcr\x19(x\x14\x14\x9d[\xaf\x9d\xad\x9f\x9c\xe9]g\r\xe7:\xbc_\xac\x9f\xd3\x98y\xf6\xb5\x1f\xfc\xbaBk\xb2\x1e4\xa3s\x8c\x80\xbc\x04\x88\x17\x1b6q\xddu+\xdau\xf1B\'\xe6b\x91\x8e\xb5\xf2q\xde\xfb\xb8\xcdX\x19\xd6\x1b.q}\xdd\x13\xb5R\x12,\xbeq\xf1#-$\xac\x8f\xc1L&>\x82\ti[\xf4E8J\x8b\xbf\xe44\xf9\xd3\x99\xb6T+p\n\xc8\x88\xda\xe2O\xbf\x1dM\xc7\xf7p\x96\x02t\xe0\xb7\xa3\x19\x00\x99\xff\x03\xa1g\xd3\x01').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'flf2v']
REMOTE_MAX_IMAGES = 2

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            ram_gib = ltx_available_ram_gib() if REMOTE_IS_LTX else None
            ram_ready = ram_gib is None or ram_gib >= LTX_MIN_AVAILABLE_RAM_GIB
            if not ram_ready:
                gpu_message = (f'主機 RAM 不足：可用約 {ram_gib:.1f} GiB，'
                               f'建議至少 {LTX_MIN_AVAILABLE_RAM_GIB:.0f} GiB；'
                               '請重新啟動 Kaggle Kernel 或降低生成負載。')
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready and ram_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'ram_ready': ram_ready, 'ram_available_gib': round(ram_gib, 2) if ram_gib is not None else None,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return

        '''
        Reject new video jobs if host RAM is already critically constrained.
        主機 RAM 已不足時拒絕新影片工作，避免繼續累積記憶體壓力。
        '''
        if REMOTE_IS_LTX and ltx_available_ram_gib() < LTX_MIN_AVAILABLE_RAM_GIB:
            self.send_json(503, {'error': 'Kaggle host RAM is too low for LTX-2.5 inference'}, 30)
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                 'FPS', 'IMAGE_STRENGTH', 'MODEL_TIER', 'VAE_TILE_SIZE',
                 'VAE_TILE_OVERLAP', 'VAE_TEMPORAL_SIZE', 'VAE_TEMPORAL_OVERLAP']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            '''
            Fill missing LTX runtime constants without overwriting notebook settings.
            補齊缺少的 LTX 執行參數，但不覆蓋 Notebook 已設定的值。
            '''
            if REMOTE_IS_LTX:
                for name, value in LTX_REMOTE_DEFAULTS.items():
                    namespace.setdefault(name, value)

            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            if REMOTE_IS_LTX:
                available_gib = ltx_available_ram_gib()
                if available_gib < LTX_MIN_AVAILABLE_RAM_GIB:
                    raise RuntimeError(f'生成前主機 RAM 不足：{available_gib:.1f} GiB 可用，'
                                       f'至少需要 {LTX_MIN_AVAILABLE_RAM_GIB:.0f} GiB 緩衝。')
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 13 節完成：選用 FIFO／ngrok API 已啟動。')


14. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_LTX-2.5_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 14 節完成：API Schema 和 client 說明已顯示。')
